# Hybrid AE-MSCNN with Learned Calibrated Fusion for Binary Intrusion Detection on NSL-KDD

End-to-end pipeline for NSL-KDD binary intrusion detection using an autoencoder for reconstruction cues, a multi-scale CNN for classification, and a learned calibrator for fused decisions, with evaluation metrics and plotting.

## Imports & Setup

Core libraries, ML frameworks, metrics, and global visualization settings used throughout the pipeline.

In [2]:
from __future__ import annotations

import argparse
import contextlib
import copy
import json
import random
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_curve,
    roc_auc_score,
)
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid", context="talk")

## Embedded Scaffold (Fallback Data Loader)

If the external `nslkdd_training_scaffold` module is unavailable, this embedded block provides dataset loading, preprocessing, and leakage checks.

In [3]:
try:
    from nslkdd_training_scaffold import load_data, preprocess_data, summarize_class_distribution
except ModuleNotFoundError:
    print("[Import] nslkdd_training_scaffold not found; using embedded scaffold fallback.")
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler

    _NSL_KDD_COLUMNS: List[str] = [
        "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
        "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
        "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
        "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
        "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
        "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
        "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
        "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
        "label", "difficulty",
    ]
    _FEATURE_COLUMNS: List[str] = _NSL_KDD_COLUMNS[:41]
    _CATEGORICAL_COLUMNS: List[str] = ["protocol_type", "service", "flag"]
    _NUMERICAL_COLUMNS: List[str] = [c for c in _FEATURE_COLUMNS if c not in _CATEGORICAL_COLUMNS]
    _EXPECTED_TRAIN_ROWS = 125973
    _EXPECTED_TEST_ROWS = 22544

    @dataclass
    class _DatasetBundle:
        train_df: pd.DataFrame
        test_df: pd.DataFrame
        train_path: Path
        test_path: Path

    @dataclass
    class _PreprocessBundle:
        X_train: np.ndarray
        X_test: np.ndarray
        y_train: np.ndarray
        y_test: np.ndarray
        preprocessor: ColumnTransformer
        leakage_checks_passed: bool

    def _validate_split_paths(train_path: Path, test_path: Path) -> None:
        train_name = train_path.name.lower()
        test_name = test_path.name.lower()
        assert train_path.exists(), f"Train file does not exist: {train_path}"
        assert test_path.exists(), f"Test file does not exist: {test_path}"
        assert train_path.resolve() != test_path.resolve(), "Train and test files must be different files."
        assert "kddtrain+" in train_name, f"Expected official training file containing 'KDDTrain+'. Got: {train_path.name}"
        assert "kddtest+" in test_name, f"Expected official testing file containing 'KDDTest+'. Got: {test_path.name}"

    def _read_nsl_kdd_txt(path: Path, source_name: str) -> pd.DataFrame:
        df = pd.read_csv(path, names=_NSL_KDD_COLUMNS, header=None)
        df["label"] = df["label"].astype(str).str.strip().str.rstrip(".")
        df["__source__"] = source_name
        return df

    def load_data(dataset_dir: Path | str) -> _DatasetBundle:
        dataset_dir = Path(dataset_dir)
        train_path = dataset_dir / "KDDTrain+.txt"
        test_path = dataset_dir / "KDDTest+.txt"
        _validate_split_paths(train_path, test_path)

        train_df = _read_nsl_kdd_txt(train_path, source_name="KDDTrain+")
        test_df = _read_nsl_kdd_txt(test_path, source_name="KDDTest+")
        assert train_df.shape[0] == _EXPECTED_TRAIN_ROWS, f"Expected {_EXPECTED_TRAIN_ROWS} rows in KDDTrain+.txt, got {train_df.shape[0]}."
        assert test_df.shape[0] == _EXPECTED_TEST_ROWS, f"Expected {_EXPECTED_TEST_ROWS} rows in KDDTest+.txt, got {test_df.shape[0]}."

        return _DatasetBundle(train_df=train_df, test_df=test_df, train_path=train_path, test_path=test_path)

    def load_merged_data(dataset_dir: Path | str, merge_test_ratio: float = 0.2, seed: int = 42) -> _DatasetBundle:
        """Load official train/test, merge them, and perform stratified train-test split.
        
        Args:
            dataset_dir: Path to NSL-KDD dataset directory.
            merge_test_ratio: Ratio of merged data to use as test set (default 0.2).
            seed: Random seed for reproducible split.
        
        Returns:
            _DatasetBundle with merged and re-split train/test dataframes.
        """
        # Load the official data first
        official_data = load_data(dataset_dir)
        
        # Concatenate train and test dataframes
        merged_df = pd.concat([official_data.train_df, official_data.test_df], axis=0, ignore_index=True)
        labels_for_stratification = merged_df["label"].astype(str).to_numpy()
        
        # Perform stratified train-test split on merged data (fixed random_state for consistency across ensemble runs)
        indices = np.arange(merged_df.shape[0])
        train_indices, test_indices = train_test_split(
            indices,
            test_size=merge_test_ratio,
            random_state=42,
            stratify=labels_for_stratification,
            shuffle=True,
        )
        
        # Create new train and test dataframes from merged data
        merged_train_df = merged_df.iloc[train_indices].reset_index(drop=True)
        merged_test_df = merged_df.iloc[test_indices].reset_index(drop=True)
        
        # Return bundle with merged split (preserve original paths for logging)
        return _DatasetBundle(
            train_df=merged_train_df,
            test_df=merged_test_df,
            train_path=official_data.train_path,
            test_path=official_data.test_path,
        )

    def _build_preprocessor(categorical_encoding: str = "onehot", scaler_type: str = "standard") -> ColumnTransformer:
        if scaler_type == "standard":
            num_transformer = Pipeline(steps=[("scaler", StandardScaler())])
        elif scaler_type == "minmax":
            num_transformer = Pipeline(steps=[("scaler", MinMaxScaler())])
        else:
            raise ValueError("scaler_type must be either 'standard' or 'minmax'.")

        if categorical_encoding == "onehot":
            try:
                encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
            except TypeError:
                encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
            cat_transformer = Pipeline(steps=[("encoder", encoder)])
        elif categorical_encoding == "label":
            cat_transformer = Pipeline(
                steps=[
                    (
                        "encoder",
                        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                    )
                ]
            )
        else:
            raise ValueError("categorical_encoding must be either 'onehot' or 'label'.")

        return ColumnTransformer(
            transformers=[
                ("num", num_transformer, _NUMERICAL_COLUMNS),
                ("cat", cat_transformer, _CATEGORICAL_COLUMNS),
            ],
            remainder="drop",
        )

    def _assert_preprocessor_fit_is_train_only(
        preprocessor: ColumnTransformer,
        X_train_df: pd.DataFrame,
        scaler_type: str,
    ) -> None:
        train_num = X_train_df[_NUMERICAL_COLUMNS].to_numpy(dtype=np.float64)
        num_scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
        if scaler_type == "standard":
            assert np.allclose(num_scaler.mean_, train_num.mean(axis=0), atol=1e-10)
            assert np.allclose(num_scaler.var_, train_num.var(axis=0), atol=1e-10)
        else:
            assert np.allclose(num_scaler.data_min_, train_num.min(axis=0), atol=1e-10)
            assert np.allclose(num_scaler.data_max_, train_num.max(axis=0), atol=1e-10)

    def _count_exact_feature_label_overlap(train_df: pd.DataFrame, test_df: pd.DataFrame) -> int:
        overlap_cols = _FEATURE_COLUMNS + ["label"]
        train_hash = pd.util.hash_pandas_object(train_df[overlap_cols], index=False).to_numpy()
        test_hash = pd.util.hash_pandas_object(test_df[overlap_cols], index=False).to_numpy()
        overlap = np.intersect1d(train_hash, test_hash, assume_unique=False)
        return int(overlap.size)

    def preprocess_data(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        categorical_encoding: str = "onehot",
        scaler_type: str = "standard",
    ) -> _PreprocessBundle:
        X_train_df = train_df[_FEATURE_COLUMNS].copy()
        y_train = train_df["label"].astype(str).to_numpy()
        X_test_df = test_df[_FEATURE_COLUMNS].copy()
        y_test = test_df["label"].astype(str).to_numpy()

        preprocessor = _build_preprocessor(categorical_encoding=categorical_encoding, scaler_type=scaler_type)
        X_train = preprocessor.fit_transform(X_train_df)
        X_test = preprocessor.transform(X_test_df)
        _assert_preprocessor_fit_is_train_only(preprocessor=preprocessor, X_train_df=X_train_df, scaler_type=scaler_type)

        X_train = np.asarray(X_train, dtype=np.float32)
        X_test = np.asarray(X_test, dtype=np.float32)
        return _PreprocessBundle(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            preprocessor=preprocessor,
            leakage_checks_passed=True,
        )

    def summarize_class_distribution(train_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, object]:
        train_counts = train_df["label"].value_counts().sort_values(ascending=False)
        test_counts = test_df["label"].value_counts().sort_values(ascending=False)
        train_labels = set(train_df["label"].unique())
        test_labels = set(test_df["label"].unique())
        unknown_attack_types = sorted((test_labels - train_labels) - {"normal"})
        unknown_attack_count = int(test_df[test_df["label"].isin(unknown_attack_types)].shape[0])
        exact_overlap_count = _count_exact_feature_label_overlap(train_df, test_df)
        return {
            "train_samples": int(train_df.shape[0]),
            "test_samples": int(test_df.shape[0]),
            "train_num_classes": int(train_counts.shape[0]),
            "test_num_classes": int(test_counts.shape[0]),
            "unknown_attack_type_count": int(len(unknown_attack_types)),
            "unknown_attack_record_count": unknown_attack_count,
            "exact_feature_label_overlap_count": exact_overlap_count,
            "unknown_attack_types": unknown_attack_types,
            "train_class_distribution": train_counts.to_dict(),
            "test_class_distribution": test_counts.to_dict(),
        }


[Import] nslkdd_training_scaffold not found; using embedded scaffold fallback.


## Configuration & Argument Parsing

Command-line and notebook parameters that control dataset selection, model settings, and calibration behavior.

In [4]:
def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="NSL-KDD accuracy-only: two-branch AE + binary CNN booster + calibrator"
    )
    parser.add_argument(
        "--dataset-dir",
        type=str,
        default="",
        help=(
            "Path to folder containing KDDTrain+.txt and KDDTest+.txt. "
            "If omitted, auto-detects common local and Kaggle input locations."
        ),
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument(
        "--val-seed",
        type=int,
        default=0,
        help="Seed for train/val split (fixed across ensemble runs, independent of model seed). 0=same as --seed.",
    )
    parser.add_argument(
        "--force-cpu",
        type=int,
        default=0,
        choices=[0, 1],
        help="Force CPU execution even when CUDA appears available.",
    )

    parser.add_argument("--categorical-encoding", type=str, default="onehot", choices=["onehot", "label"])
    parser.add_argument("--scaler", type=str, default="standard", choices=["standard", "minmax"])

    parser.add_argument("--smote-k-neighbors", type=int, default=5)
    parser.add_argument(
        "--two-branch-system",
        type=int,
        default=1,
        choices=[1],
        help="Two-branch system is fixed to the locked configuration.",
    )
    parser.add_argument(
        "--use-binary-cnn-with-ae-booster",
        "--use_binary_cnn_with_ae_booster",
        type=int,
        default=1,
        choices=[1],
        help="Binary CNN + AE booster mode is fixed to the locked configuration.",
    )
    parser.add_argument(
        "--binary-booster-use-calibrator",
        "--binary_booster_use_calibrator",
        type=int,
        default=1,
        choices=[1],
        help=(
            "Fit the post-hoc MLP calibrator on the validation holdout using "
            "[cnn_attack_prob, ae_q_score, cnn_entropy] and standardized reconstruction stats."
        ),
    )
    parser.add_argument(
        "--binary-booster-decision-policy",
        type=str,
        default="calibrated",
        choices=["calibrated"],
        help="Binary booster inference policy is fixed to the calibrated MLP path.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-max-normal-fpr",
        type=float,
        default=0.12,
        help=(
            "Constraint used when selecting calibrator decision threshold on validation holdout."
        ),
    )
    parser.add_argument(
        "--binary-booster-calibrator-threshold-steps",
        type=int,
        default=401,
        help="Number of thresholds scanned in [0,1] for constrained calibrator decision selection.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-c",
        type=float,
        default=1.0,
        help="Inverse regularization strength C mapped to AdamW weight decay (weight_decay=1/C) for MLP calibrator.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-class-weight-balanced",
        type=int,
        default=1,
        choices=[1],
        help="Use class-weighted BCE balancing in MLP calibrator fitting.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-include-recon-stats",
        "--binary_booster_calibrator_include_recon_stats",
        type=int,
        default=1,
        choices=[1],
        help=(
            "Include standardized AE reconstruction stats [mse, mae, max_abs] as additional inputs."
        ),
    )
    parser.add_argument(
        "--binary-booster-calibrator-hidden-dim",
        "--binary_booster_calibrator_hidden_dim",
        type=int,
        default=16,
        help="Hidden dimension for the two-layer MLP calibrator gate.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-dropout",
        "--binary_booster_calibrator_dropout",
        type=float,
        default=0.3,
        help="Dropout probability used inside the MLP calibrator gate.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-epochs",
        "--binary_booster_calibrator_epochs",
        type=int,
        default=80,
        help="Maximum number of MLP calibrator epochs (early stopping may end sooner).",
    )
    parser.add_argument(
        "--binary-booster-calibrator-lr",
        "--binary_booster_calibrator_lr",
        type=float,
        default=5e-4,
        help="Learning rate for AdamW in MLP calibrator training.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-patience",
        "--binary_booster_calibrator_patience",
        type=int,
        default=10,
        help="Early-stopping patience for MLP calibrator validation loss.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-batch-size",
        "--binary_booster_calibrator_batch_size",
        type=int,
        default=1024,
        help="Batch size used for MLP calibrator training and inference.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-optimize",
        "--binary_booster_calibrator_optimize",
        type=int,
        default=1,
        choices=[1],
        help="Enable calibrator-only grid optimization on the validation holdout.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-max-normal-fpr-grid",
        "--binary_booster_calibrator_max_normal_fpr_grid",
        type=str,
        default="0.10,0.12,0.14,0.16",
        help=(
            "Comma-separated grid of max-normal-FPR constraints scanned when calibrator optimization is enabled."
        ),
    )
    parser.add_argument(
        "--binary-booster-calibrator-include-recon-stats-grid",
        "--binary_booster_calibrator_include_recon_stats_grid",
        type=str,
        default="1",
        help=(
            "Comma-separated grid over recon-stat feature inclusion for calibrator optimization."
        ),
    )
    parser.add_argument(
        "--binary-booster-calibrator-temperature-grid",
        "--binary_booster_calibrator_temperature_grid",
        type=str,
        default="1.0",
        help=(
            "Comma-separated probability-temperature grid for calibrator optimization."
        ),
    )
    parser.add_argument(
        "--binary-booster-calibrator-opt-objective",
        "--binary_booster_calibrator_opt_objective",
        type=str,
        default="recall_balanced",
        choices=["recall_balanced"],
        help="Primary objective used to rank calibrator-optimization candidates on validation holdout.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-opt-target-normal-fpr",
        "--binary_booster_calibrator_opt_target_normal_fpr",
        type=float,
        default=0.12,
        help="Target normal-FPR anchor used in calibrator optimization scoring.",
    )
    parser.add_argument(
        "--binary-booster-calibrator-opt-target-fpr-weight",
        "--binary_booster_calibrator_opt_target_fpr_weight",
        type=float,
        default=0.5,
        help="Penalty weight for excess FPR in optimizer ranking.",
    )
    parser.add_argument(
        "--two-branch-fusion-mode",
        type=str,
        default="binary_ae_booster",
        choices=["binary_ae_booster"],
        help="Kept for compatibility; only binary_ae_booster is supported in accuracy-only mode.",
    )

    parser.add_argument("--latent-dim", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=512)

    parser.add_argument("--ae-epochs", type=int, default=30)
    parser.add_argument("--ae-lr", type=float, default=1e-3)
    parser.add_argument("--ae-patience", type=int, default=30)
    parser.add_argument("--ae-dropout", type=float, default=0.2)

    parser.add_argument("--cnn-epochs", type=int, default=30)
    parser.add_argument("--cnn-lr", type=float, default=8e-4)
    parser.add_argument("--cnn-patience", type=int, default=30)
    parser.add_argument("--cnn-dropout", type=float, default=0.25)
    parser.add_argument(
        "--cnn-variant",
        type=str,
        default="multiscale_stem",
        choices=["multiscale_stem"],
        help="CNN backbone variant used by the locked configuration.",
    )
    parser.add_argument("--cnn-base-channels", type=int, default=32)
    parser.add_argument("--cnn-stem-dim", type=int, default=64)
    parser.add_argument(
        "--enable-ae-latent-fusion",
        type=int,
        default=1,
        choices=[1],
        help="AE latent fusion is fixed to the locked configuration.",
    )
    parser.add_argument(
        "--enable-residual-vector-fusion",
        "--enable_residual_vector_fusion",
        type=int,
        default=1,
        choices=[1],
        help="Concatenate standardized absolute residual vectors |X - AE(X)| to CNN inputs.",
    )
    parser.add_argument(
        "--residual-vector-weight",
        "--residual_vector_weight",
        type=float,
        default=1.0,
        help="Scaling weight applied to standardized residual vector features before CNN concatenation.",
    )
    parser.add_argument(
        "--cnn-num-classes",
        type=int,
        default=2,
        choices=[2],
        help="Binary head fixed to 2 classes.",
    )
    parser.add_argument("--label-smoothing", type=float, default=0.02)

    parser.add_argument("--recon-percentile", type=float, default=94.2)

    parser.add_argument(
        "--merge-official-splits",
        "--merge_official_splits",
        type=int,
        default=0,
        choices=[0, 1],
        help=(
            "If set to 1, merge KDDTrain+ and KDDTest+ into a single dataset and perform "
            "a stratified train-test split using --merge-test-ratio. "
            "If set to 0 (default), use the official split."
        ),
    )
    parser.add_argument(
        "--merge-test-ratio",
        "--merge_test_ratio",
        type=float,
        default=0.2,
        help=(
            "When --merge-official-splits=1, controls the ratio of merged data to use as test set. "
            "Default 0.2 (20%% test, 80%% train). Ignored if --merge-official-splits=0."
        ),
    )
    parser.add_argument(
        "--ensemble-seeds",
        "--ensemble_seeds",
        type=str,
        default="",
        help=(
            "Comma-separated list of seeds for ensemble averaging. If empty, single run. "
            "Example: '42,123,456' will train 3 models and average calibrated probabilities."
        ),
    )

    parser.add_argument(
        "--max-train-samples",
        type=int,
        default=0,
        help="0 means full training set. Use a smaller value for quick smoke runs.",
    )
    parser.add_argument(
        "--output-dir",
        type=str,
        default="training_runs",
        help="Directory where each run will save train.log, test_results.json, and run_config.json.",
    )

    return parser

## Data Loading & Preprocessing

Dataset resolution helpers and path utilities used before calling the scaffold/fallback loaders and preprocessing routines in `run_pipeline`.

In [5]:
def _has_nslkdd_txt_pair(dataset_dir: Path) -> bool:
    return (dataset_dir / "KDDTrain+.txt").exists() and (dataset_dir / "KDDTest+.txt").exists()


def resolve_dataset_dir(dataset_dir_arg: str) -> Path:
    raw_arg = str(dataset_dir_arg).strip()

    if raw_arg:
        provided = Path(raw_arg).expanduser()
        if _has_nslkdd_txt_pair(provided):
            return provided
        nested = provided / "nsl-kdd"
        if _has_nslkdd_txt_pair(nested):
            return nested
        raise FileNotFoundError(
            "Could not find KDDTrain+.txt and KDDTest+.txt under provided dataset-dir: "
            f"{provided}"
        )

    candidates = [
        Path("NSL-KDD"),
        Path("NSL-KDD") / "nsl-kdd",
        Path("nsl-kdd"),
        Path("nslkdd"),
        Path("/kaggle/input/datasets/hassan06/nslkdd"),
        Path("/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd"),
        Path("/kaggle/input/nslkdd"),
        Path("/kaggle/input/nslkdd/nsl-kdd"),
        Path("/kaggle/input/nsl-kdd"),
        Path("/kaggle/input/nsl-kdd/nsl-kdd"),
    ]
    for candidate in candidates:
        if _has_nslkdd_txt_pair(candidate):
            return candidate

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob("KDDTrain+.txt"):
            parent = train_path.parent
            if _has_nslkdd_txt_pair(parent):
                return parent

    searched = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not auto-detect NSL-KDD dataset directory. Pass --dataset-dir explicitly. "
        "Expected a folder containing KDDTrain+.txt and KDDTest+.txt.\n"
        f"Searched candidates:\n{searched}"
    )


def resolve_output_dir(output_dir_arg: str) -> Path:
    output_dir = Path(str(output_dir_arg).strip() or "training_runs").expanduser()
    if output_dir.is_absolute():
        return output_dir
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists():
        return kaggle_working / output_dir
    return output_dir

## Label Preparation

Binary label construction and unknown attack detection for the NSL-KDD split.

In [6]:
@dataclass
class LabelBundle:
    y_train: np.ndarray
    y_test: np.ndarray
    unknown_attack_mask_test: np.ndarray
    unknown_attack_types: List[str]


def build_binary_labels(train_raw_labels: np.ndarray, test_raw_labels: np.ndarray) -> LabelBundle:
    train_labels = train_raw_labels.astype(str)
    test_labels = test_raw_labels.astype(str)

    y_train = (train_labels != "normal").astype(np.int64)
    y_test = (test_labels != "normal").astype(np.int64)

    train_attack_types = set(train_labels) - {"normal"}
    test_attack_types = set(test_labels) - {"normal"}
    unknown_attack_types = sorted(test_attack_types - train_attack_types)
    unknown_attack_mask_test = np.isin(test_labels, unknown_attack_types)

    return LabelBundle(
        y_train=y_train,
        y_test=y_test,
        unknown_attack_mask_test=unknown_attack_mask_test,
        unknown_attack_types=unknown_attack_types,
    )

## Train/Val Split and Borderline-SMOTE

Stratified train/validation split and minority class augmentation via Borderline-SMOTE.

In [7]:
def train_val_split(
    X_train: np.ndarray,
    y_train: np.ndarray,
    val_ratio: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    indices = np.arange(X_train.shape[0])
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_ratio,
        random_state=seed,
        stratify=y_train,
        shuffle=True,
    )

    # Ensure split partitions are disjoint and complete.
    assert len(set(train_idx).intersection(set(val_idx))) == 0, "Train/val overlap detected."
    assert len(train_idx) + len(val_idx) == len(indices), "Train/val partition mismatch."

    return X_train[train_idx], X_train[val_idx], y_train[train_idx], y_train[val_idx], train_idx, val_idx


def apply_borderline_smote(
    X_train_split: np.ndarray,
    y_train_split: np.ndarray,
    seed: int,
    desired_k_neighbors: int = 5,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, int]]:
    # Lazy import reduces heavy SciPy-related startup overhead at script import time.
    from imblearn.over_sampling import BorderlineSMOTE

    class_counts = {int(cls): int(cnt) for cls, cnt in zip(*np.unique(y_train_split, return_counts=True))}
    min_count = min(class_counts.values())
    effective_k = max(1, min(desired_k_neighbors, min_count - 1))

    smote = BorderlineSMOTE(
        random_state=seed,
        k_neighbors=effective_k,
        kind="borderline-1",
    )
    X_res, y_res = smote.fit_resample(X_train_split, y_train_split)

    # Hard leakage rule: only train-split data goes into fit_resample.
    assert X_res.shape[0] >= X_train_split.shape[0], "Unexpected SMOTE output size reduction."

    stats = {
        "before_class_0": class_counts.get(0, 0),
        "before_class_1": class_counts.get(1, 0),
        "after_class_0": int(np.sum(y_res == 0)),
        "after_class_1": int(np.sum(y_res == 1)),
        "k_neighbors": int(effective_k),
    }
    return X_res.astype(np.float32), y_res.astype(np.int64), stats

## Autoencoder Definition & Training

Autoencoder architecture plus training and reconstruction feature extraction utilities.

In [8]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 32, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


def _make_loader(
    X: np.ndarray,
    y: np.ndarray | None,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    X_tensor = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.long)
        ds = TensorDataset(X_tensor, y_tensor)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_autoencoder(
    model: Autoencoder,
    X_train_ae_input: np.ndarray,
    X_val: np.ndarray,
    device: torch.device,
    epochs: int,
    batch_size: int,
    lr: float,
    patience: int,
) -> Autoencoder:
    train_loader = _make_loader(X_train_ae_input, None, batch_size=batch_size, shuffle=True)
    val_loader = _make_loader(X_val, None, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses: List[float] = []
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        val_losses: List[float] = []
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)
                recon = model(xb)
                val_loss = criterion(recon, xb)
                val_losses.append(float(val_loss.detach().cpu().item()))

        mean_train = float(np.mean(train_losses)) if train_losses else float("inf")
        mean_val = float(np.mean(val_losses)) if val_losses else float("inf")

        print(f"[AE] Epoch {epoch:03d} | train_mse={mean_train:.6f} | val_mse={mean_val:.6f}")

        if mean_val < best_val_loss:
            best_val_loss = mean_val
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[AE] Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model


def encode_features(
    model: Autoencoder,
    X: np.ndarray,
    device: torch.device,
    batch_size: int = 2048,
) -> np.ndarray:
    loader = _make_loader(X, None, batch_size=batch_size, shuffle=False)
    model.eval()

    chunks: List[np.ndarray] = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encode(xb)
            chunks.append(z.detach().cpu().numpy())

    return np.concatenate(chunks, axis=0).astype(np.float32)


def reconstruction_error_features_and_weighted(
    model: Autoencoder,
    X: np.ndarray,
    feature_std: np.ndarray,
    device: torch.device,
    batch_size: int = 2048,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    loader = _make_loader(X, None, batch_size=batch_size, shuffle=False)
    model.eval()

    errs: List[np.ndarray] = []
    feats: List[np.ndarray] = []
    wmse_chunks: List[np.ndarray] = []
    feature_std_safe = np.where(np.asarray(feature_std, dtype=np.float32) < 1e-6, 1.0, feature_std).astype(np.float32)
    feature_std_tensor = torch.tensor(feature_std_safe, dtype=torch.float32, device=device)

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            recon = model(xb)
            residual = xb - recon

            mse = torch.mean(residual ** 2, dim=1)
            mae = torch.mean(torch.abs(residual), dim=1)
            max_abs = torch.max(torch.abs(residual), dim=1).values
            errs.append(mse.detach().cpu().numpy())
            feats.append(torch.stack([mse, mae, max_abs], dim=1).detach().cpu().numpy())

            squared_error = residual ** 2
            weighted = squared_error / (feature_std_tensor ** 2)
            wmse = torch.mean(weighted, dim=1)
            wmse_chunks.append(wmse.detach().cpu().numpy())

    return (
        np.concatenate(errs, axis=0).astype(np.float32),
        np.concatenate(feats, axis=0).astype(np.float32),
        np.concatenate(wmse_chunks, axis=0).astype(np.float32),
    )


def residual_vector_features(
    model: Autoencoder,
    X: np.ndarray,
    device: torch.device,
    batch_size: int = 2048,
    absolute: bool = True,   # Keep signature but override behavior
) -> np.ndarray:
    loader = _make_loader(X, None, batch_size=batch_size, shuffle=False)
    model.eval()
    residual_chunks: List[np.ndarray] = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            recon = model(xb)
            residual = (xb - recon) ** 2   # Always squared; ignore 'absolute' flag
            residual_chunks.append(residual.detach().cpu().numpy())
    return np.concatenate(residual_chunks, axis=0).astype(np.float32)


def fit_feature_standardizer(features: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = np.mean(features, axis=0).astype(np.float32)
    std = np.std(features, axis=0).astype(np.float32)
    std = np.where(std < 1e-8, 1.0, std).astype(np.float32)
    return mean, std


def apply_feature_standardizer(features: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((features - mean) / std).astype(np.float32)

## Multi-Scale CNN Definition

Multi-scale 1D CNN backbone with residual blocks and squeeze-excitation used for binary intrusion classification.

In [9]:
class SqueezeExcite1D(nn.Module):
    def __init__(self, channels: int, reduction: int = 8) -> None:
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Conv1d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv1d(hidden, channels, kernel_size=1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.pool(x)
        s = self.relu(self.fc1(s))
        s = self.sigmoid(self.fc2(s))
        return x * s


class ResidualConv1DBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.25) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SqueezeExcite1D(out_channels)
        self.drop = nn.Dropout(dropout)
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.relu(out + identity)
        return out


class MultiScaleCNN1DClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int = 2,
        dropout: float = 0.25,
        base_channels: int = 32,
        use_stem: bool = False,
        stem_dim: int = 64,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.use_stem = use_stem
        self.sequence_dim = int(stem_dim if use_stem else input_dim)
        if self.use_stem:
            self.stem = nn.Sequential(
                nn.Linear(input_dim, self.sequence_dim),
                nn.BatchNorm1d(self.sequence_dim),
                nn.ReLU(),
                #nn.Dropout(dropout),
            )
        else:
            self.stem = nn.Identity()

        self.branch_k3 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(),
        )
        self.branch_k5 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(),
        )
        self.branch_k7 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(),
        )

        merged_channels = base_channels * 3
        self.post_merge = nn.Sequential(
            nn.Conv1d(merged_channels, merged_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(merged_channels),
            nn.ReLU(),
        )
        self.block1 = ResidualConv1DBlock(merged_channels, base_channels * 4, dropout=dropout)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.block2 = ResidualConv1DBlock(base_channels * 4, base_channels * 4, dropout=dropout)

        self.gmax = nn.AdaptiveMaxPool1d(1)
        self.gavg = nn.AdaptiveAvgPool1d(1)
        head_dim = base_channels * 8
        self.classifier = nn.Sequential(
            nn.Linear(head_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_stem:
            x = self.stem(x)
        x = x.view(x.shape[0], 1, self.sequence_dim)
        x = torch.cat([self.branch_k3(x), self.branch_k5(x), self.branch_k7(x)], dim=1)
        x = self.post_merge(x)
        x = self.block1(x)
        x = self.pool(x)
        x = self.block2(x)
        x = torch.cat([self.gmax(x).squeeze(-1), self.gavg(x).squeeze(-1)], dim=1)
        return self.classifier(x)


def build_cnn_classifier(
    input_dim: int,
    num_classes: int,
    dropout: float,
    stem_dim: int,
    base_channels: int,
) -> nn.Module:
    return MultiScaleCNN1DClassifier(
        input_dim=input_dim,
        num_classes=num_classes,
        dropout=dropout,
        base_channels=base_channels,
        use_stem=True,
        stem_dim=stem_dim,
    )

## CNN Training

Training loop with cross-entropy loss, AdamW optimizer, learning-rate scheduler, gradient clipping, and early stopping.

In [10]:
def train_cnn_classifier(
    model: nn.Module,
    X_train_latent: np.ndarray,
    y_train: np.ndarray,
    X_val_latent: np.ndarray,
    y_val: np.ndarray,
    device: torch.device,
    epochs: int,
    batch_size: int,
    lr: float,
    patience: int,
    label_smoothing: float,
) -> Tuple[nn.Module, Dict[str, Any]]:
    train_loader = _make_loader(X_train_latent, y_train, batch_size=batch_size, shuffle=True)
    val_loader = _make_loader(X_val_latent, y_val, batch_size=batch_size, shuffle=False)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-7 )

    best_state = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    train_loss_history: List[float] = []
    val_loss_history: List[float] = []
    val_f1_history: List[float] = []
    train_acc_history: List[float] = []
    val_acc_history: List[float] = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses: List[float] = []
        train_true: List[np.ndarray] = []
        train_pred: List[np.ndarray] = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu().item()))
            preds = torch.argmax(logits, dim=1)
            train_true.append(yb.detach().cpu().numpy())
            train_pred.append(preds.detach().cpu().numpy())

        model.eval()
        val_losses: List[float] = []
        val_true: List[np.ndarray] = []
        val_pred: List[np.ndarray] = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                preds = torch.argmax(logits, dim=1)
                val_losses.append(float(loss.detach().cpu().item()))
                val_true.append(yb.detach().cpu().numpy())
                val_pred.append(preds.detach().cpu().numpy())

        y_true = np.concatenate(val_true, axis=0)
        y_pred_cls = np.concatenate(val_pred, axis=0)
        train_y_true = np.concatenate(train_true, axis=0) if train_true else np.array([])
        train_y_pred = np.concatenate(train_pred, axis=0) if train_pred else np.array([])
        mean_train = float(np.mean(train_losses)) if train_losses else float("inf")
        mean_val = float(np.mean(val_losses)) if val_losses else float("inf")
        train_acc = float(accuracy_score(train_y_true, train_y_pred)) if train_y_true.size > 0 else 0.0
        val_acc = float(accuracy_score(y_true, y_pred_cls)) if y_true.size > 0 else 0.0
        val_f1 = float(f1_score(y_true, y_pred_cls, average="macro", zero_division=0))
        train_loss_history.append(mean_train)
        val_loss_history.append(mean_val)
        val_f1_history.append(val_f1)
        train_acc_history.append(train_acc)
        val_acc_history.append(val_acc)
        print(
            f"[CNN] Epoch {epoch:03d} | train_loss={mean_train:.6f} | "
            f"val_loss={mean_val:.6f} | train_acc={train_acc:.4f} | "
            f"val_acc={val_acc:.4f} | val_macro_f1={val_f1:.4f}"
        )

        scheduler.step(mean_val)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[CNN] Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    history = {
        "train_loss": train_loss_history,
        "val_loss": val_loss_history,
        "val_f1": val_f1_history,
        "train_acc": train_acc_history,
        "val_acc": val_acc_history,
        "best_epoch": int(best_epoch),
    }
    return model, history

## Prediction Functions

Helpers for CNN probability inference and AE-assisted binary decision fusion.

In [11]:
def predict_prob_attack(
    model: nn.Module,
    X_latent: np.ndarray,
    device: torch.device,
    batch_size: int = 2048,
) -> np.ndarray:
    loader = _make_loader(X_latent, None, batch_size=batch_size, shuffle=False)
    model.eval()
    prob_chunks: List[np.ndarray] = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            logits = model(xb)
            prob_tensor = torch.softmax(logits, dim=1)
            p = prob_tensor[:, 1]
            prob_chunks.append(p.detach().cpu().numpy())

    return np.concatenate(prob_chunks, axis=0)



def binary_cnn_with_ae_booster_predict(
    prob_attack: np.ndarray,
    recon_errors: np.ndarray,
    normal_error_reference_sorted: np.ndarray,
    ae_attack_veto_q: float = 0.96,
    cls_threshold: float = 0.50,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    q_scores = np.clip(
        _percentile_scores_from_reference(normal_error_reference_sorted, recon_errors),
        0.0,
        1.0,
    ).astype(np.float32)

    ae_attack_veto_q = float(np.clip(ae_attack_veto_q, 0.0, 1.0))
    cls_threshold = float(np.clip(cls_threshold, 0.0, 1.0))

    y_pred_cnn = (prob_attack >= cls_threshold).astype(np.int64)
    ae_veto_attack_mask = q_scores >= ae_attack_veto_q

    y_pred = y_pred_cnn.copy()
    y_pred[ae_veto_attack_mask] = 1

    prob_attack_fused = prob_attack.astype(np.float32).copy()
    if np.any(ae_veto_attack_mask):
        prob_attack_fused[ae_veto_attack_mask] = np.maximum(
            prob_attack_fused[ae_veto_attack_mask],
            q_scores[ae_veto_attack_mask],
        )

    stats = {
        "ae_veto_attack_rate": float(np.mean(ae_veto_attack_mask)),
        "cnn_attack_rate_before_veto": float(np.mean(y_pred_cnn == 1)),
        "pred_attack_rate": float(np.mean(y_pred == 1)),
        "pred_normal_rate": float(np.mean(y_pred == 0)),
        "mean_prob_attack": float(np.mean(prob_attack.astype(np.float32))),
        "mean_q_score": float(np.mean(q_scores)),
        "ae_attack_veto_q": float(ae_attack_veto_q),
        "cnn_binary_threshold": float(cls_threshold),
    }
    return y_pred, prob_attack_fused.astype(np.float32), stats

## Calibration Feature Builders

Feature construction for the calibrator using CNN probabilities, AE q-scores, entropy, and reconstruction statistics.

In [12]:
def apply_probability_temperature(prob_attack: np.ndarray, temperature: float) -> np.ndarray:
    p = np.clip(np.asarray(prob_attack, dtype=np.float32), 1e-6, 1.0 - 1e-6)
    t = float(max(1e-3, temperature))
    if abs(t - 1.0) < 1e-6:
        return p.astype(np.float32)
    logits = np.log(p) - np.log1p(-p)
    scaled_logits = logits / t
    p_scaled = 1.0 / (1.0 + np.exp(-scaled_logits))
    return np.clip(p_scaled, 1e-6, 1.0 - 1e-6).astype(np.float32)


def parse_csv_float_grid(csv_text: str, default_values: List[float]) -> List[float]:
    text = str(csv_text).strip()
    if not text:
        return [float(v) for v in default_values]
    values: List[float] = []
    seen: set[float] = set()
    for token in text.split(","):
        part = token.strip()
        if not part:
            continue
        value = float(part)
        if value not in seen:
            seen.add(value)
            values.append(value)
    return values if values else [float(v) for v in default_values]


def parse_csv_binary_grid(csv_text: str, default_values: List[int]) -> List[int]:
    text = str(csv_text).strip()
    if not text:
        return [1 if int(v) == 1 else 0 for v in default_values]
    values: List[int] = []
    seen: set[int] = set()
    for token in text.split(","):
        part = token.strip()
        if not part:
            continue
        value = 1 if int(float(part)) == 1 else 0
        if value not in seen:
            seen.add(value)
            values.append(value)
    return values if values else [1 if int(v) == 1 else 0 for v in default_values]


def build_binary_booster_calibration_features(
    prob_attack: np.ndarray,
    recon_errors: np.ndarray,
    normal_error_reference_sorted: np.ndarray,
    temperature: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    p_attack = apply_probability_temperature(prob_attack, temperature=temperature)
    q_scores = np.clip(
        _percentile_scores_from_reference(normal_error_reference_sorted, recon_errors),
        0.0,
        1.0,
    ).astype(np.float32)
    entropy = (-(p_attack * np.log(p_attack) + (1.0 - p_attack) * np.log(1.0 - p_attack))).astype(np.float32)
    features = np.stack([p_attack, q_scores, entropy], axis=1).astype(np.float32)
    return features, q_scores, entropy


def _percentile_scores_from_reference(reference_sorted: np.ndarray, values: np.ndarray) -> np.ndarray:
    if reference_sorted.size == 0:
        return np.zeros_like(values, dtype=np.float32)
    positions = np.searchsorted(reference_sorted, values, side="right")
    return (positions / float(reference_sorted.size)).astype(np.float32)


def select_constrained_binary_threshold(
    y_true: np.ndarray,
    prob_attack: np.ndarray,
    max_normal_fpr: float,
    steps: int,
) -> Dict[str, float]:
    max_normal_fpr = float(np.clip(max_normal_fpr, 0.0, 1.0))
    steps = int(max(11, steps))
    thresholds = np.linspace(0.0, 1.0, steps)
    attack_prevalence = float(np.mean(y_true == 1))

    best_feasible: Dict[str, float] | None = None
    best_any: Dict[str, float] | None = None
    feasible_count = 0

    for threshold in thresholds:
        y_pred = (prob_attack >= float(threshold)).astype(np.int64)
        metrics = _binary_metrics_from_pred(y_true, y_pred)

        feasible_rank = (
            float(metrics["attack_recall"]),
            float(metrics["macro_f1"]),
            float(metrics["accuracy"]),
            -abs(float(metrics["pred_attack_rate"]) - attack_prevalence),
            -float(metrics["normal_fpr"]),
        )
        fpr_penalty = max(0.0, float(metrics["normal_fpr"]) - max_normal_fpr)
        fallback_rank = (
            float(metrics["attack_recall"]) - 3.0 * fpr_penalty,
            float(metrics["macro_f1"]),
            float(metrics["accuracy"]),
            -float(metrics["normal_fpr"]),
        )

        candidate_base = {
            "threshold": float(threshold),
            "accuracy": float(metrics["accuracy"]),
            "macro_f1": float(metrics["macro_f1"]),
            "attack_recall": float(metrics["attack_recall"]),
            "normal_fpr": float(metrics["normal_fpr"]),
            "pred_attack_rate": float(metrics["pred_attack_rate"]),
        }

        if float(metrics["normal_fpr"]) <= max_normal_fpr:
            feasible_count += 1
            candidate = dict(candidate_base)
            candidate["_rank"] = feasible_rank
            if best_feasible is None or feasible_rank > best_feasible["_rank"]:
                best_feasible = candidate

        candidate_any = dict(candidate_base)
        candidate_any["_rank"] = fallback_rank
        if best_any is None or fallback_rank > best_any["_rank"]:
            best_any = candidate_any

    selected = best_feasible if best_feasible is not None else best_any
    assert selected is not None, "Threshold selection failed to produce a candidate."
    selected = dict(selected)
    selected.pop("_rank", None)
    selected["max_normal_fpr_constraint"] = float(max_normal_fpr)
    selected["feasible_candidate_count"] = float(feasible_count)
    selected["used_constraint"] = float(1.0 if best_feasible is not None else 0.0)
    return selected

## Calibrator MLP Definition & Training

Two-layer MLP gate for calibrated fusion, including class-weighted training and optimization routines.

In [13]:
class CalibratorMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 16, dropout: float = 0.3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)


def _calibrator_sample_weights(y: np.ndarray, class_weight_balanced: bool) -> Tuple[np.ndarray, Dict[str, float]]:
    y_int = y.astype(np.int64)
    n_total = int(y_int.shape[0])
    n_pos = int(np.sum(y_int == 1))
    n_neg = int(np.sum(y_int == 0))
    if class_weight_balanced and n_pos > 0 and n_neg > 0:
        w_pos = float(n_total / (2.0 * n_pos))
        w_neg = float(n_total / (2.0 * n_neg))
    else:
        w_pos = 1.0
        w_neg = 1.0
    weights = np.where(y_int == 1, w_pos, w_neg).astype(np.float32)
    return weights, {"positive": w_pos, "negative": w_neg}


def predict_binary_booster_calibrator_prob(
    model: CalibratorMLP,
    X: np.ndarray,
    batch_size: int = 4096,
    device: torch.device | None = None,
) -> np.ndarray:
    model.eval()
    infer_device = device
    if infer_device is None:
        infer_device = next(model.parameters()).device

    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    probs: List[np.ndarray] = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(infer_device)
            p = model(xb)
            probs.append(p.detach().cpu().numpy())
    return np.clip(np.concatenate(probs, axis=0), 1e-6, 1.0 - 1e-6).astype(np.float32)


def fit_binary_booster_mlp_calibrator(
    y_val: np.ndarray,
    prob_attack_val: np.ndarray,
    recon_errors_val: np.ndarray,
    normal_error_reference_sorted: np.ndarray,
    max_normal_fpr: float,
    threshold_steps: int,
    c_value: float,
    class_weight_balanced: bool,
    recon_feature_stats_val: np.ndarray | None = None,
    include_recon_feature_stats: bool = False,
    temperature: float = 1.0,
    device: torch.device | None = None,
    hidden_dim: int = 16,
    dropout: float = 0.3,
    epochs: int = 80,
    lr: float = 5e-4,
    patience: int = 10,
    batch_size: int = 1024,
    calibrator_context: Dict[str, Any] | None = None,
) -> Tuple[
    CalibratorMLP,
    float,
    np.ndarray,
    np.ndarray,
    Dict[str, Any],
    Dict[str, float],
]:
    X_cal, q_scores, entropy = build_binary_booster_calibration_features(
        prob_attack=prob_attack_val,
        recon_errors=recon_errors_val,
        normal_error_reference_sorted=normal_error_reference_sorted,
        temperature=float(max(1e-3, temperature)),
    )

    feature_order: List[str] = ["cnn_attack_probability", "ae_q_score", "cnn_entropy"]
    recon_feature_mean: np.ndarray | None = None
    recon_feature_std: np.ndarray | None = None
    include_recon = bool(include_recon_feature_stats and recon_feature_stats_val is not None)
    if include_recon:
        recon_val = np.asarray(recon_feature_stats_val, dtype=np.float32)
        recon_feature_mean, recon_feature_std = fit_feature_standardizer(recon_val)
        recon_val_std = apply_feature_standardizer(recon_val, recon_feature_mean, recon_feature_std)
        X_cal = np.concatenate([X_cal, recon_val_std], axis=1).astype(np.float32)
        if recon_val_std.shape[1] == 3:
            feature_order.extend(["recon_mse_z", "recon_mae_z", "recon_max_abs_z"])
        elif recon_val_std.shape[1] == 4:
            feature_order.extend(["recon_mse_z", "recon_mae_z", "recon_max_abs_z", "recon_weighted_mse_z"])
        elif recon_val_std.shape[1] > 4:
            feature_order.extend(["recon_mse_z", "recon_mae_z", "recon_max_abs_z", "recon_weighted_mse_z"])
            feature_order.extend([f"residual_pca_{i}" for i in range(recon_val_std.shape[1] - 4)])
        else:
            feature_order.extend([f"recon_stat_{i}_z" for i in range(recon_val_std.shape[1])])

    y_cal = y_val.astype(np.int64)
    indices = np.arange(X_cal.shape[0])
    try:
        train_idx, stop_idx = train_test_split(
            indices,
            test_size=0.2,
            random_state=0,
            stratify=y_cal,
            shuffle=True,
        )
    except ValueError:
        train_idx, stop_idx = indices, indices

    X_train_cal = X_cal[train_idx].astype(np.float32)
    y_train_cal = y_cal[train_idx].astype(np.float32)
    X_stop_cal = X_cal[stop_idx].astype(np.float32)
    y_stop_cal = y_cal[stop_idx].astype(np.float32)

    weights_train_np, class_weight_values = _calibrator_sample_weights(
        y_train_cal.astype(np.int64),
        class_weight_balanced=bool(class_weight_balanced),
    )
    weights_stop_np, _ = _calibrator_sample_weights(
        y_stop_cal.astype(np.int64),
        class_weight_balanced=bool(class_weight_balanced),
    )

    cal_device = device if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = CalibratorMLP(
        input_dim=int(X_cal.shape[1]),
        hidden_dim=int(hidden_dim),
        dropout=float(dropout),
    ).to(cal_device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=float(max(1e-6, lr)),
        weight_decay=float(1.0 / max(1e-6, c_value)),
    )
    criterion = nn.BCELoss(reduction="none")

    x_train_t = torch.tensor(X_train_cal, dtype=torch.float32, device=cal_device)
    y_train_t = torch.tensor(y_train_cal, dtype=torch.float32, device=cal_device)
    w_train_t = torch.tensor(weights_train_np, dtype=torch.float32, device=cal_device)
    x_stop_t = torch.tensor(X_stop_cal, dtype=torch.float32, device=cal_device)
    y_stop_t = torch.tensor(y_stop_cal, dtype=torch.float32, device=cal_device)
    w_stop_t = torch.tensor(weights_stop_np, dtype=torch.float32, device=cal_device)

    best_state = copy.deepcopy(model.state_dict())
    best_stop_loss = float("inf")
    best_epoch = 0
    epochs_no_improve = 0

    batch_size = int(max(32, batch_size))
    num_train = x_train_t.shape[0]
    for epoch in range(1, int(max(1, epochs)) + 1):
        model.train()
        order = torch.randperm(num_train, device=cal_device)
        batch_losses: List[float] = []
        for start in range(0, num_train, batch_size):
            idx = order[start:start + batch_size]
            xb = x_train_t[idx]
            yb = y_train_t[idx]
            wb = w_train_t[idx]

            optimizer.zero_grad()
            p = model(xb)
            loss = (criterion(p, yb) * wb).mean()
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        with torch.no_grad():
            p_stop = model(x_stop_t)
            stop_loss = float((criterion(p_stop, y_stop_t) * w_stop_t).mean().detach().cpu().item())

        if stop_loss < best_stop_loss:
            best_stop_loss = stop_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= int(max(1, patience)):
                break

    model.load_state_dict(best_state)

    prob_cal = predict_binary_booster_calibrator_prob(
        model=model,
        X=X_cal,
        batch_size=batch_size,
        device=cal_device,
    )
    selection = select_constrained_binary_threshold(
        y_true=y_val,
        prob_attack=prob_cal,
        max_normal_fpr=float(max_normal_fpr),
        steps=int(threshold_steps),
    )
    selected_threshold = float(selection["threshold"])
    y_pred_cal = (prob_cal >= selected_threshold).astype(np.int64)
    selected_metrics = _binary_metrics_from_pred(y_val, y_pred_cal)

    calibrator_info: Dict[str, Any] = {
        "enabled": True,
        "model": "two_layer_mlp_gate",
        "feature_order": feature_order,
        "c_value": float(max(1e-6, c_value)),
        "weight_decay": float(1.0 / max(1e-6, c_value)),
        "class_weight_balanced": bool(class_weight_balanced),
        "class_weights": class_weight_values,
        "hidden_dim": int(hidden_dim),
        "dropout": float(dropout),
        "epochs": int(max(1, epochs)),
        "lr": float(max(1e-6, lr)),
        "patience": int(max(1, patience)),
        "batch_size": int(batch_size),
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_stop_loss),
        "calibrator_device": str(cal_device),
        "include_recon_feature_stats": bool(include_recon),
        "temperature": float(max(1e-3, temperature)),
        "recon_feature_mean": (recon_feature_mean.astype(float).tolist() if recon_feature_mean is not None else []),
        "recon_feature_std": (recon_feature_std.astype(float).tolist() if recon_feature_std is not None else []),
        "selected_threshold": float(selected_threshold),
        "max_normal_fpr_constraint": float(max_normal_fpr),
        "selection": selection,
        "validation_metrics_at_selected_threshold": {
            "accuracy": float(selected_metrics["accuracy"]),
            "macro_f1": float(selected_metrics["macro_f1"]),
            "attack_recall": float(selected_metrics["attack_recall"]),
            "normal_fpr": float(selected_metrics["normal_fpr"]),
            "pred_attack_rate": float(selected_metrics["pred_attack_rate"]),
        },
    }
    if calibrator_context is not None:
        calibrator_info["calibrator_context"] = _to_serializable(calibrator_context)

    decision_stats = {
        "mean_prob_attack_feature": float(np.mean(X_cal[:, 0])),
        "mean_q_score": float(np.mean(q_scores)),
        "mean_entropy": float(np.mean(entropy)),
        "mean_calibrated_probability": float(np.mean(prob_cal)),
        "calibrator_best_validation_loss": float(best_stop_loss),
        "selected_threshold": float(selected_threshold),
        "selected_attack_recall": float(selected_metrics["attack_recall"]),
        "selected_normal_fpr": float(selected_metrics["normal_fpr"]),
        "temperature": float(max(1e-3, temperature)),
    }
    if include_recon:
        decision_stats["mean_recon_feature_stats"] = np.mean(
            np.asarray(recon_feature_stats_val, dtype=np.float32), axis=0
        ).astype(float).tolist()
    return model, selected_threshold, prob_cal, y_pred_cal, calibrator_info, decision_stats


def optimize_binary_booster_mlp_calibrator(
    y_val: np.ndarray,
    prob_attack_val: np.ndarray,
    recon_errors_val: np.ndarray,
    normal_error_reference_sorted: np.ndarray,
    threshold_steps: int,
    c_value: float,
    class_weight_balanced: bool,
    recon_feature_stats_val: np.ndarray | None,
    max_normal_fpr_grid: List[float],
    include_recon_feature_stats_grid: List[int],
    temperature_grid: List[float],
    selection_objective: str = "accuracy",
    target_normal_fpr: float = 0.12,
    target_normal_fpr_weight: float = 0.0,
    device: torch.device | None = None,
    hidden_dim: int = 16,
    dropout: float = 0.3,
    epochs: int = 80,
    lr: float = 5e-4,
    patience: int = 10,
    batch_size: int = 1024,
    calibrator_context: Dict[str, Any] | None = None,
) -> Tuple[
    CalibratorMLP,
    float,
    np.ndarray,
    np.ndarray,
    Dict[str, Any],
    Dict[str, float],
]:
    selection_objective = str(selection_objective).strip().lower()
    if selection_objective not in {"accuracy", "macro_f1", "attack_recall", "recall_balanced"}:
        selection_objective = "accuracy"
    target_normal_fpr = float(np.clip(target_normal_fpr, 0.0, 1.0))
    target_normal_fpr_weight = float(max(0.0, target_normal_fpr_weight))

    def _objective_score(metrics: Dict[str, float]) -> float:
        if selection_objective == "macro_f1":
            return float(metrics["macro_f1"])
        if selection_objective == "attack_recall":
            return float(metrics["attack_recall"])
        if selection_objective == "recall_balanced":
            return 0.5 * (
                float(metrics["attack_recall"]) + (1.0 - float(metrics["normal_fpr"]))
            )
        return float(metrics["accuracy"])

    best_rank: Tuple[float, float, float, float, float] | None = None
    best_outputs: Tuple[
        CalibratorMLP,
        float,
        np.ndarray,
        np.ndarray,
        Dict[str, Any],
        Dict[str, float],
    ] | None = None
    candidates: List[Dict[str, Any]] = []

    for max_fpr in max_normal_fpr_grid:
        for include_recon in include_recon_feature_stats_grid:
            for temperature in temperature_grid:
                (
                    model,
                    threshold,
                    prob_cal,
                    y_pred_cal,
                    calibrator_info,
                    decision_stats,
                ) = fit_binary_booster_mlp_calibrator(
                    y_val=y_val,
                    prob_attack_val=prob_attack_val,
                    recon_errors_val=recon_errors_val,
                    normal_error_reference_sorted=normal_error_reference_sorted,
                    max_normal_fpr=float(max_fpr),
                    threshold_steps=int(threshold_steps),
                    c_value=float(c_value),
                    class_weight_balanced=bool(class_weight_balanced),
                    recon_feature_stats_val=recon_feature_stats_val,
                    include_recon_feature_stats=bool(int(include_recon) == 1),
                    temperature=float(max(1e-3, temperature)),
                    device=device,
                    hidden_dim=int(hidden_dim),
                    dropout=float(dropout),
                    epochs=int(epochs),
                    lr=float(lr),
                    patience=int(patience),
                    batch_size=int(batch_size),
                    calibrator_context=calibrator_context,
                )
                metrics = _binary_metrics_from_pred(y_val, y_pred_cal)
                objective_score = _objective_score(metrics)
                # Anchor should discourage exceeding target FPR; lower-than-target FPR should not be penalized.
                fpr_anchor_penalty = max(0.0, float(metrics["normal_fpr"]) - target_normal_fpr)
                adjusted_objective_score = objective_score - target_normal_fpr_weight * fpr_anchor_penalty
                rank = (
                    float(adjusted_objective_score),
                    float(objective_score),
                    float(metrics["macro_f1"]),
                    float(metrics["attack_recall"]),
                    -float(metrics["normal_fpr"]),
                )
                candidate = {
                    "max_normal_fpr": float(max_fpr),
                    "include_recon_feature_stats": bool(int(include_recon) == 1),
                    "temperature": float(max(1e-3, temperature)),
                    "selected_threshold": float(threshold),
                    "accuracy": float(metrics["accuracy"]),
                    "macro_f1": float(metrics["macro_f1"]),
                    "attack_recall": float(metrics["attack_recall"]),
                    "normal_fpr": float(metrics["normal_fpr"]),
                    "objective": str(selection_objective),
                    "objective_score": float(objective_score),
                    "fpr_anchor_penalty": float(fpr_anchor_penalty),
                    "adjusted_objective_score": float(adjusted_objective_score),
                }
                candidates.append(candidate)

                if best_rank is None or rank > best_rank:
                    best_rank = rank
                    best_outputs = (
                        model,
                        threshold,
                        prob_cal,
                        y_pred_cal,
                        calibrator_info,
                        decision_stats,
                    )

    assert best_outputs is not None, "Calibrator optimization did not produce any candidate."
    (
        best_model,
        best_threshold,
        best_prob_cal,
        best_y_pred_cal,
        best_calibrator_info,
        best_decision_stats,
    ) = best_outputs

    best_calibrator_info = dict(best_calibrator_info)
    best_calibrator_info["optimization"] = {
        "enabled": True,
        "search_mode": "calibrator_only",
        "selection_objective": str(selection_objective),
        "target_normal_fpr": float(target_normal_fpr),
        "target_normal_fpr_weight": float(target_normal_fpr_weight),
        "candidate_count": int(len(candidates)),
        "max_normal_fpr_grid": [float(v) for v in max_normal_fpr_grid],
        "include_recon_feature_stats_grid": [int(v) for v in include_recon_feature_stats_grid],
        "temperature_grid": [float(max(1e-3, v)) for v in temperature_grid],
        "candidates": candidates,
        "selected": {
            "max_normal_fpr": float(best_calibrator_info.get("max_normal_fpr_constraint", 0.0)),
            "include_recon_feature_stats": bool(best_calibrator_info.get("include_recon_feature_stats", False)),
            "temperature": float(best_calibrator_info.get("temperature", 1.0)),
            "selected_threshold": float(best_threshold),
        },
    }
    return (
        best_model,
        best_threshold,
        best_prob_cal,
        best_y_pred_cal,
        best_calibrator_info,
        best_decision_stats,
    )

## Evaluation Metrics & Helper Functions

Metric bundles, confusion matrix computation, and reference-baseline comparison utilities.

In [14]:
@dataclass
class EvaluationBundle:
    accuracy: float
    known_subset_accuracy: float
    unknown_subset_accuracy: float
    balanced_accuracy: float
    macro_f1: float
    precision_macro: float
    recall_macro: float
    precision_attack: float
    recall_attack: float
    f1_attack: float
    roc_auc: float
    pr_auc: float
    known_attack_recall: float
    unknown_attack_recall: float
    normal_false_positive_rate: float
    confusion: np.ndarray


@dataclass
class RunArtifacts:
    run_id: str
    run_dir: Path
    log_path: Path
    results_path: Path
    config_path: Path


REFERENCE_BASELINES: Dict[str, Dict[str, float]] = {
    "aggressive_mode_reference": {
        "unknown_attack_recall": 0.9995,
        "normal_false_positive_rate": 0.29,
    },
    "balanced_mode_reference": {
        "unknown_attack_recall": 0.85,
        "normal_false_positive_rate": 0.14,
    },
}


def evaluation_bundle_to_dict(eval_bundle: EvaluationBundle) -> Dict[str, Any]:
    return {
        "accuracy": float(eval_bundle.accuracy),
        "known_subset_accuracy": float(eval_bundle.known_subset_accuracy),
        "unknown_subset_accuracy": float(eval_bundle.unknown_subset_accuracy),
        "balanced_accuracy": float(eval_bundle.balanced_accuracy),
        "macro_precision": float(eval_bundle.precision_macro),
        "macro_recall": float(eval_bundle.recall_macro),
        "macro_f1": float(eval_bundle.macro_f1),
        "attack_precision": float(eval_bundle.precision_attack),
        "attack_recall": float(eval_bundle.recall_attack),
        "attack_f1": float(eval_bundle.f1_attack),
        "roc_auc": float(eval_bundle.roc_auc),
        "pr_auc": float(eval_bundle.pr_auc),
        "known_attack_recall": float(eval_bundle.known_attack_recall),
        "unknown_attack_recall": float(eval_bundle.unknown_attack_recall),
        "normal_false_positive_rate": float(eval_bundle.normal_false_positive_rate),
        "confusion_matrix": eval_bundle.confusion,
    }


def build_reference_run_comparison(eval_bundle: EvaluationBundle) -> Dict[str, Dict[str, float]]:
    current_unknown = float(eval_bundle.unknown_attack_recall)
    current_fpr = float(eval_bundle.normal_false_positive_rate)

    comparison: Dict[str, Dict[str, float]] = {}
    for ref_name, ref_values in REFERENCE_BASELINES.items():
        ref_unknown = float(ref_values["unknown_attack_recall"])
        ref_fpr = float(ref_values["normal_false_positive_rate"])
        comparison[ref_name] = {
            "reference_unknown_attack_recall": ref_unknown,
            "reference_normal_false_positive_rate": ref_fpr,
            "current_unknown_attack_recall": current_unknown,
            "current_normal_false_positive_rate": current_fpr,
            "delta_unknown_attack_recall": current_unknown - ref_unknown,
            "delta_normal_false_positive_rate": current_fpr - ref_fpr,
        }
    return comparison


def print_reference_run_comparison(comparison: Dict[str, Dict[str, float]]) -> None:
    print("\n=== Comparison To Historical Baselines ===")
    for ref_name, vals in comparison.items():
        print(
            f"{ref_name}: "
            f"unknown_recall_delta={vals['delta_unknown_attack_recall']:+.4f} | "
            f"normal_fpr_delta={vals['delta_normal_false_positive_rate']:+.4f}"
        )


class TeeStream:
    def __init__(self, *streams: Any) -> None:
        self.streams = streams

    def write(self, data: str) -> int:
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self) -> None:
        for stream in self.streams:
            stream.flush()


def _to_serializable(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_serializable(v) for v in value]
    return value


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(_to_serializable(payload), handle, indent=2)


def create_run_artifacts(output_dir: Path) -> RunArtifacts:
    output_dir.mkdir(parents=True, exist_ok=True)
    for _ in range(10):
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        run_dir = output_dir / f"run_{run_id}"
        if not run_dir.exists():
            run_dir.mkdir(parents=True, exist_ok=False)
            return RunArtifacts(
                run_id=run_id,
                run_dir=run_dir,
                log_path=run_dir / "train.log",
                results_path=run_dir / "test_results.json",
                config_path=run_dir / "run_config.json",
            )
    raise RuntimeError("Could not create a unique run directory for artifacts.")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_torch_device(force_cpu: bool = False) -> torch.device:
    if force_cpu:
        print("[Device] Forced CPU mode enabled (--force-cpu 1).")
        return torch.device("cpu")

    if not torch.cuda.is_available():
        return torch.device("cpu")

    try:
        major, minor = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
    except Exception as exc:
        print(f"[Device] Could not query CUDA device details ({exc}); falling back to CPU.")
        return torch.device("cpu")

    # Newer PyTorch wheels may drop support for older GPUs (e.g., sm_60 on P100).
    if major < 7:
        print(
            "[Device] CUDA device is available but unsupported by this PyTorch build: "
            f"{name} (sm_{major}{minor}). Falling back to CPU."
        )
        return torch.device("cpu")

    try:
        _ = torch.tensor([0.0], device="cuda")
    except Exception as exc:
        print(f"[Device] CUDA runtime probe failed ({exc}); falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cuda")


def _binary_metrics_from_pred(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    attack_mask = y_true == 1
    normal_mask = y_true == 0
    attack_recall = float(np.mean(y_pred[attack_mask] == 1)) if np.any(attack_mask) else 0.0
    normal_fpr = float(np.mean(y_pred[normal_mask] == 1)) if np.any(normal_mask) else 0.0
    pred_attack_rate = float(np.mean(y_pred == 1))
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "attack_recall": attack_recall,
        "normal_fpr": normal_fpr,
        "pred_attack_rate": pred_attack_rate,
    }


def evaluate_binary(
    y_true: np.ndarray,
    prob_attack: np.ndarray,
    recon_errors_test: np.ndarray,
    recon_threshold: float,
    unknown_attack_mask_test: np.ndarray,
    cls_threshold: float,
    precomputed_pred: np.ndarray | None = None,
) -> EvaluationBundle:
    if precomputed_pred is None:
        y_pred = np.logical_or(
            prob_attack >= float(cls_threshold),
            recon_errors_test >= float(recon_threshold),
        ).astype(np.int64)
    else:
        y_pred = precomputed_pred.astype(np.int64)

    acc = float(accuracy_score(y_true, y_pred))
    known_subset_mask = ~unknown_attack_mask_test
    known_subset_acc = float(accuracy_score(y_true[known_subset_mask], y_pred[known_subset_mask]))
    unknown_subset_acc = (
        float(accuracy_score(y_true[unknown_attack_mask_test], y_pred[unknown_attack_mask_test]))
        if np.any(unknown_attack_mask_test)
        else 0.0
    )
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_attack, r_attack, f_attack, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    if len(np.unique(y_true)) > 1:
        roc_auc = float(roc_auc_score(y_true, prob_attack))
        pr_auc = float(average_precision_score(y_true, prob_attack))
    else:
        roc_auc = 0.0
        pr_auc = 0.0

    known_attack_mask = (y_true == 1) & (~unknown_attack_mask_test)
    unknown_attack_recall = (
        float(np.mean(y_pred[unknown_attack_mask_test] == 1)) if np.any(unknown_attack_mask_test) else 0.0
    )
    known_attack_recall = float(np.mean(y_pred[known_attack_mask] == 1)) if np.any(known_attack_mask) else 0.0

    normal_mask = y_true == 0
    normal_fpr = float(np.mean(y_pred[normal_mask] == 1)) if np.any(normal_mask) else 0.0

    conf = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return EvaluationBundle(
        accuracy=acc,
        known_subset_accuracy=known_subset_acc,
        unknown_subset_accuracy=unknown_subset_acc,
        balanced_accuracy=bal_acc,
        macro_f1=float(f1),
        precision_macro=float(precision),
        recall_macro=float(recall),
        precision_attack=float(p_attack),
        recall_attack=float(r_attack),
        f1_attack=float(f_attack),
        roc_auc=roc_auc,
        pr_auc=pr_auc,
        known_attack_recall=known_attack_recall,
        unknown_attack_recall=unknown_attack_recall,
        normal_false_positive_rate=normal_fpr,
        confusion=conf,
    )


def print_eval(eval_bundle: EvaluationBundle, title: str = "Test Metrics (Binary)") -> None:
    print(f"\n=== {title} ===")
    print(f"Accuracy (All Test):       {eval_bundle.accuracy:.4f}")
    print(f"Accuracy (Known Subset):   {eval_bundle.known_subset_accuracy:.4f}")
    print(f"Accuracy (Unknown Subset): {eval_bundle.unknown_subset_accuracy:.4f}")
    print(f"Balanced Accuracy:         {eval_bundle.balanced_accuracy:.4f}")
    print(f"Macro Precision:           {eval_bundle.precision_macro:.4f}")
    print(f"Macro Recall:              {eval_bundle.recall_macro:.4f}")
    print(f"Macro F1:                  {eval_bundle.macro_f1:.4f}")
    print(f"Attack Precision:          {eval_bundle.precision_attack:.4f}")
    print(f"Attack Recall:             {eval_bundle.recall_attack:.4f}")
    print(f"Attack F1:                 {eval_bundle.f1_attack:.4f}")
    print(f"ROC-AUC:                   {eval_bundle.roc_auc:.4f}")
    print(f"PR-AUC:                    {eval_bundle.pr_auc:.4f}")
    print(f"Known Attack Recall:       {eval_bundle.known_attack_recall:.4f}")
    print(f"Unknown Attack Recall:     {eval_bundle.unknown_attack_recall:.4f}")
    print(f"Normal False PositiveRate: {eval_bundle.normal_false_positive_rate:.4f}")
    print("Confusion Matrix [[TN, FP], [FN, TP]]:")
    print(eval_bundle.confusion)

## Main Pipeline Run Function

Orchestrates data loading, training, calibration, evaluation, and artifact collection for a single run.

In [15]:
def run_pipeline(args: argparse.Namespace) -> Dict[str, Any]:
    set_seed(args.seed)

    dataset_dir = Path(args.dataset_dir)
    print(f"[Data] Dataset directory: {dataset_dir}")
    
    # Load data using either official split or merged split
    if bool(args.merge_official_splits == 1):
        print(f"[Data] Using merged-official-splits mode with test ratio {args.merge_test_ratio}")
        data = load_merged_data(dataset_dir, merge_test_ratio=args.merge_test_ratio, seed=args.seed)
    else:
        print("[Data] Using official KDDTrain+/KDDTest+ split")
        data = load_data(dataset_dir)
    
    dist = summarize_class_distribution(data.train_df, data.test_df)

    print("=== Split and Unknown Attack Summary ===")
    print(f"Train rows: {dist['train_samples']}")
    print(f"Test rows:  {dist['test_samples']}")
    print(f"Unknown attack types in test: {dist['unknown_attack_type_count']}")
    print(f"Unknown attack records in test: {dist['unknown_attack_record_count']}")

    prep = preprocess_data(
        data.train_df,
        data.test_df,
        categorical_encoding=args.categorical_encoding,
        scaler_type=args.scaler,
    )
    assert prep.leakage_checks_passed, "Base preprocessing leakage checks failed."

    labels = build_binary_labels(
        train_raw_labels=data.train_df["label"].to_numpy(),
        test_raw_labels=data.test_df["label"].to_numpy(),
    )
    test_raw_labels = data.test_df["label"].astype(str).to_numpy()

    X_train = prep.X_train
    y_train = labels.y_train
    X_test = prep.X_test
    y_test = labels.y_test

    if args.max_train_samples > 0 and args.max_train_samples < X_train.shape[0]:
        idx = np.random.RandomState(args.seed).choice(X_train.shape[0], size=args.max_train_samples, replace=False)
        X_train = X_train[idx]
        y_train = y_train[idx]
        print(f"[Info] Using train subset for faster iteration: {X_train.shape[0]} samples")

    # Use val_seed for train/val split (fixed across ensemble runs for consistent validation)
    effective_val_seed = args.val_seed if args.val_seed != 0 else args.seed
    X_train_split, X_val, y_train_split, y_val, _, _ = train_val_split(
        X_train,
        y_train,
        val_ratio=args.val_ratio,
        seed=effective_val_seed,
    )

    device = resolve_torch_device(force_cpu=bool(args.force_cpu == 1))
    print(f"\nUsing device: {device}")

    two_branch_enabled = True
    use_binary_cnn_with_ae_booster = True
    enable_ae_latent_fusion = True
    enable_residual_vector_fusion = bool(args.enable_residual_vector_fusion == 1)
    cnn_num_classes = 2
    effective_cnn_variant = "multiscale_stem"

    default_fpr_grid = [0.11, float(args.binary_booster_calibrator_max_normal_fpr), 0.13]
    binary_booster_calibrator_max_fpr_grid = parse_csv_float_grid(
        args.binary_booster_calibrator_max_normal_fpr_grid,
        default_values=default_fpr_grid,
    )
    binary_booster_calibrator_include_recon_stats_grid = parse_csv_binary_grid(
        args.binary_booster_calibrator_include_recon_stats_grid,
        default_values=[int(args.binary_booster_calibrator_include_recon_stats)],
    )
    binary_booster_calibrator_temperature_grid = parse_csv_float_grid(
        args.binary_booster_calibrator_temperature_grid,
        default_values=[1.0],
    )
    binary_booster_calibrator_optimize = bool(args.binary_booster_calibrator_optimize == 1)
    binary_ae_booster_q_threshold = 0.96

    print("[Mode] AE latent fusion enabled: concatenating AE latent vectors to CNN inputs.")
    if enable_residual_vector_fusion:
        print(
            "[Mode] Residual vector fusion enabled: concatenating standardized |X - AE(X)| vectors to CNN input "
            f"(weight={float(args.residual_vector_weight):.3f})."
        )
    print(
        "[Mode] Binary CNN + AE booster enabled: CNN is binary (normal/attack) and "
        f"AE forces attack when q >= {binary_ae_booster_q_threshold:.2f}."
    )
    if args.binary_booster_use_calibrator == 1 and binary_booster_calibrator_optimize:
        print(
            "[Mode] Calibrator-only optimization enabled: "
            f"fpr_grid={binary_booster_calibrator_max_fpr_grid}, "
            f"recon_stats_grid={binary_booster_calibrator_include_recon_stats_grid}, "
            f"temperature_grid={binary_booster_calibrator_temperature_grid}, "
            f"objective={str(args.binary_booster_calibrator_opt_objective)}, "
            f"target_fpr={float(args.binary_booster_calibrator_opt_target_normal_fpr):.3f}, "
            f"target_fpr_weight={float(args.binary_booster_calibrator_opt_target_fpr_weight):.3f}."
        )

    normal_train_mask = y_train_split == 0
    normal_val_mask = y_val == 0
    X_train_normal = X_train_split[normal_train_mask]
    if X_train_normal.shape[0] < 2:
        raise RuntimeError("Two-branch mode requires at least two normal samples in train split.")

    X_val_normal = X_val[normal_val_mask] if np.any(normal_val_mask) else X_val
    X_train_cnn, y_train_smote, smote_stats = apply_borderline_smote(
        X_train_split,
        y_train_split,
        seed=args.seed,
        desired_k_neighbors=args.smote_k_neighbors,
    )

    autoencoder = Autoencoder(
        input_dim=X_train_split.shape[1],
        latent_dim=args.latent_dim,
        dropout=args.ae_dropout,
    ).to(device)
    autoencoder = train_autoencoder(
        model=autoencoder,
        X_train_ae_input=X_train_normal.astype(np.float32),
        X_val=X_val_normal,
        device=device,
        epochs=args.ae_epochs,
        batch_size=args.batch_size,
        lr=args.ae_lr,
        patience=args.ae_patience,
    )

    # After training autoencoder, compute feature-wise std on normal training data.
    normal_features = X_train_normal.astype(np.float32)
    feature_std = np.std(normal_features, axis=0).astype(np.float32)
    feature_std = np.where(feature_std < 1e-6, 1.0, feature_std)  # avoid division by zero

    calibrator_context: Dict[str, Any] = {
        "feature_std": feature_std,
        "feature_std_min": float(np.min(feature_std)),
        "feature_std_max": float(np.max(feature_std)),
        "feature_std_mean": float(np.mean(feature_std)),
    }

    calibrator_residual_pca_val = np.zeros((X_val.shape[0], 0), dtype=np.float32)
    calibrator_residual_pca_test = np.zeros((X_test.shape[0], 0), dtype=np.float32)
    calibrator_residual_pca_components = 0
    residual_train_split_raw_for_cal: np.ndarray | None = None
    residual_val_raw_for_cal: np.ndarray | None = None
    residual_test_raw_for_cal: np.ndarray | None = None
    if args.binary_booster_use_calibrator == 1:
        residual_train_split_raw_for_cal = residual_vector_features(
            autoencoder,
            X_train_split.astype(np.float32),
            device=device,
        )
        residual_val_raw_for_cal = residual_vector_features(
            autoencoder,
            X_val.astype(np.float32),
            device=device,
        )
        residual_test_raw_for_cal = residual_vector_features(
            autoencoder,
            X_test.astype(np.float32),
            device=device,
        )
        calibrator_residual_pca_components = int(min(16, residual_train_split_raw_for_cal.shape[1]))
        if calibrator_residual_pca_components > 0:
            residual_pca = PCA(n_components=calibrator_residual_pca_components, random_state=args.seed)
            _ = residual_pca.fit_transform(residual_train_split_raw_for_cal)
            calibrator_residual_pca_val = residual_pca.transform(residual_val_raw_for_cal).astype(np.float32)
            calibrator_residual_pca_test = residual_pca.transform(residual_test_raw_for_cal).astype(np.float32)
            calibrator_context["residual_pca_n_components"] = int(calibrator_residual_pca_components)
            calibrator_context["residual_pca_explained_variance_ratio"] = residual_pca.explained_variance_ratio_.astype(float).tolist()
            calibrator_context["residual_pca_explained_variance_total"] = float(np.sum(residual_pca.explained_variance_ratio_))

    latent_val_ae = encode_features(autoencoder, X_val.astype(np.float32), device=device)
    latent_test_ae = encode_features(autoencoder, X_test.astype(np.float32), device=device)

    
    
    _, val_recon_feat, val_weighted_mse = reconstruction_error_features_and_weighted(
        model=autoencoder,
        X=X_val,
        feature_std=feature_std,
        device=device,
    )
    val_recon_err = val_weighted_mse.astype(np.float32)
    val_recon_feat = np.concatenate([val_recon_feat, val_weighted_mse.reshape(-1, 1)], axis=1).astype(np.float32)
    if calibrator_residual_pca_val.shape[1] > 0:
        val_recon_feat = np.concatenate([val_recon_feat, calibrator_residual_pca_val], axis=1).astype(np.float32)
    val_recon_err_normal = val_recon_err[normal_val_mask] if np.any(normal_val_mask) else val_recon_err
    normal_error_reference_sorted = np.sort(val_recon_err_normal.astype(np.float32))
    recon_threshold = float(np.percentile(val_recon_err_normal, args.recon_percentile))

    X_train_cnn_raw = X_train_cnn.astype(np.float32)
    X_val_cnn_raw = X_val.astype(np.float32)
    X_test_cnn_raw = X_test.astype(np.float32)

    X_train_cnn_parts: List[np.ndarray] = [X_train_cnn_raw]
    X_val_cnn_parts: List[np.ndarray] = [X_val_cnn_raw]
    X_test_cnn_parts: List[np.ndarray] = [X_test_cnn_raw]

    latent_train_for_cnn = encode_features(autoencoder, X_train_cnn_raw, device=device)
    X_train_cnn_parts.append(latent_train_for_cnn)
    X_val_cnn_parts.append(latent_val_ae)
    X_test_cnn_parts.append(latent_test_ae)

    residual_vector_dim = 0
    if enable_residual_vector_fusion:
        residual_train_raw = residual_vector_features(autoencoder, X_train_cnn_raw, device=device)
        if residual_val_raw_for_cal is not None:
            residual_val_raw = residual_val_raw_for_cal
        else:
            residual_val_raw = residual_vector_features(autoencoder, X_val_cnn_raw, device=device)
        if residual_test_raw_for_cal is not None:
            residual_test_raw = residual_test_raw_for_cal
        else:
            residual_test_raw = residual_vector_features(autoencoder, X_test_cnn_raw, device=device)
        if residual_train_split_raw_for_cal is not None:
            residual_train_split_raw = residual_train_split_raw_for_cal
        else:
            residual_train_split_raw = residual_vector_features(
                autoencoder,
                X_train_split.astype(np.float32),
                device=device,
            )

        residual_vec_mean, residual_vec_std = fit_feature_standardizer(residual_train_raw)
        residual_weight = float(args.residual_vector_weight)

        residual_train = (
            apply_feature_standardizer(residual_train_raw, residual_vec_mean, residual_vec_std) * residual_weight
        ).astype(np.float32)
        residual_val = (
            apply_feature_standardizer(residual_val_raw, residual_vec_mean, residual_vec_std) * residual_weight
        ).astype(np.float32)
        residual_test = (
            apply_feature_standardizer(residual_test_raw, residual_vec_mean, residual_vec_std) * residual_weight
        ).astype(np.float32)

        X_train_cnn_parts.append(residual_train)
        X_val_cnn_parts.append(residual_val)
        X_test_cnn_parts.append(residual_test)
        residual_vector_dim = int(residual_train.shape[1])

    X_train_cnn = np.concatenate(X_train_cnn_parts, axis=1).astype(np.float32)
    X_val_cnn = np.concatenate(X_val_cnn_parts, axis=1).astype(np.float32)
    X_test_cnn_prepared = np.concatenate(X_test_cnn_parts, axis=1).astype(np.float32)

    print(f"\n=== Borderline-SMOTE Stats (classifier branch raw train split features) ===")
    print(f"Class0 before: {smote_stats['before_class_0']} | after: {smote_stats['after_class_0']}")
    print(f"Class1 before: {smote_stats['before_class_1']} | after: {smote_stats['after_class_1']}")
    print(f"k_neighbors used: {smote_stats['k_neighbors']}")

    cnn = build_cnn_classifier(
        input_dim=X_train_cnn.shape[1],
        num_classes=cnn_num_classes,
        dropout=args.cnn_dropout,
        stem_dim=args.cnn_stem_dim,
        base_channels=args.cnn_base_channels,
    ).to(device)
    print(
        f"[CNN] Variant={effective_cnn_variant} | input_dim={X_train_cnn.shape[1]} | "
        f"num_classes={cnn_num_classes} | base_channels={args.cnn_base_channels} | stem_dim={args.cnn_stem_dim}"
    )

    cnn, cnn_history = train_cnn_classifier(
        model=cnn,
        X_train_latent=X_train_cnn,
        y_train=y_train_smote,
        X_val_latent=X_val_cnn,
        y_val=y_val,
        device=device,
        epochs=args.cnn_epochs,
        batch_size=args.batch_size,
        lr=args.cnn_lr,
        patience=args.cnn_patience,
        label_smoothing=args.label_smoothing,
    )

    prob_attack_val = predict_prob_attack(cnn, X_val_cnn, device=device)
    cls_threshold = 0.5

    y_pred_val_fixed, _, val_decision_stats_fixed = binary_cnn_with_ae_booster_predict(
        prob_attack=prob_attack_val,
        recon_errors=val_recon_err,
        normal_error_reference_sorted=normal_error_reference_sorted,
        ae_attack_veto_q=float(binary_ae_booster_q_threshold),
        cls_threshold=float(cls_threshold),
    )
    val_fused_metrics_fixed = _binary_metrics_from_pred(y_val, y_pred_val_fixed)

    binary_booster_calibrator_model: CalibratorMLP | None = None
    binary_booster_calibrator_threshold = 0.5
    binary_booster_calibrator_info: Dict[str, Any] = {}
    binary_booster_active_policy = "fixed_rule"
    y_pred_test_calibrated: np.ndarray | None = None
    
    # Initialize calibrated probabilities with raw values (may be overwritten by calibrator)
    prob_attack_val_calibrated = prob_attack_val.astype(np.float32)

    if args.binary_booster_use_calibrator == 1:
        if binary_booster_calibrator_optimize:
            (
                binary_booster_calibrator_model,
                binary_booster_calibrator_threshold,
                prob_attack_val_calibrated,
                y_pred_val_calibrated,
                binary_booster_calibrator_info,
                val_decision_stats_calibrated,
            ) = optimize_binary_booster_mlp_calibrator(
                y_val=y_val,
                prob_attack_val=prob_attack_val,
                recon_errors_val=val_recon_err,
                normal_error_reference_sorted=normal_error_reference_sorted,
                threshold_steps=int(args.binary_booster_calibrator_threshold_steps),
                c_value=float(args.binary_booster_calibrator_c),
                class_weight_balanced=bool(args.binary_booster_calibrator_class_weight_balanced == 1),
                recon_feature_stats_val=val_recon_feat,
                max_normal_fpr_grid=binary_booster_calibrator_max_fpr_grid,
                include_recon_feature_stats_grid=binary_booster_calibrator_include_recon_stats_grid,
                temperature_grid=binary_booster_calibrator_temperature_grid,
                selection_objective=str(args.binary_booster_calibrator_opt_objective),
                target_normal_fpr=float(args.binary_booster_calibrator_opt_target_normal_fpr),
                target_normal_fpr_weight=float(args.binary_booster_calibrator_opt_target_fpr_weight),
                device=device,
                hidden_dim=int(args.binary_booster_calibrator_hidden_dim),
                dropout=float(args.binary_booster_calibrator_dropout),
                epochs=int(args.binary_booster_calibrator_epochs),
                lr=float(args.binary_booster_calibrator_lr),
                patience=int(args.binary_booster_calibrator_patience),
                batch_size=int(args.binary_booster_calibrator_batch_size),
                calibrator_context=calibrator_context,
            )
        else:
            (
                binary_booster_calibrator_model,
                binary_booster_calibrator_threshold,
                prob_attack_val_calibrated,
                y_pred_val_calibrated,
                binary_booster_calibrator_info,
                val_decision_stats_calibrated,
            ) = fit_binary_booster_mlp_calibrator(
                y_val=y_val,
                prob_attack_val=prob_attack_val,
                recon_errors_val=val_recon_err,
                normal_error_reference_sorted=normal_error_reference_sorted,
                max_normal_fpr=float(args.binary_booster_calibrator_max_normal_fpr),
                threshold_steps=int(args.binary_booster_calibrator_threshold_steps),
                c_value=float(args.binary_booster_calibrator_c),
                class_weight_balanced=bool(args.binary_booster_calibrator_class_weight_balanced == 1),
                recon_feature_stats_val=val_recon_feat,
                include_recon_feature_stats=bool(args.binary_booster_calibrator_include_recon_stats == 1),
                temperature=1.0,
                device=device,
                hidden_dim=int(args.binary_booster_calibrator_hidden_dim),
                dropout=float(args.binary_booster_calibrator_dropout),
                epochs=int(args.binary_booster_calibrator_epochs),
                lr=float(args.binary_booster_calibrator_lr),
                patience=int(args.binary_booster_calibrator_patience),
                batch_size=int(args.binary_booster_calibrator_batch_size),
                calibrator_context=calibrator_context,
            )

        _ = _binary_metrics_from_pred(y_val, y_pred_val_calibrated)
        binary_booster_active_policy = "calibrated_mlp"
    else:
        val_decision_stats_calibrated = {}

    two_branch_info: Dict[str, Any] = {
        "enabled": True,
        "ae_train_normal_samples": int(X_train_normal.shape[0]),
        "val_normal_samples": int(np.sum(normal_val_mask)),
        "recon_reference": "validation_normal_subset",
        "effective_two_branch_fusion_mode": "binary_ae_booster",
        "fusion_parameters": {
            "mode": "binary_cnn_with_ae_booster",
            "active_policy": str(binary_booster_active_policy),
            "ae_attack_veto_q": float(binary_ae_booster_q_threshold),
            "cnn_binary_threshold": float(cls_threshold),
            "residual_vector_fusion_enabled": bool(enable_residual_vector_fusion),
            "residual_vector_weight": float(args.residual_vector_weight),
        },
        "validation_fusion_metrics_fixed_rule": val_fused_metrics_fixed,
        "validation_decision_stats_fixed_rule": val_decision_stats_fixed,
    }
    if binary_booster_calibrator_model is not None:
        two_branch_info["binary_booster_calibrator"] = binary_booster_calibrator_info
        two_branch_info["validation_decision_stats_calibrated"] = val_decision_stats_calibrated

    if binary_booster_active_policy == "calibrated_mlp" and binary_booster_calibrator_model is not None:
        two_branch_info["validation_fusion_metrics"] = _binary_metrics_from_pred(y_val, y_pred_val_calibrated)
        two_branch_info["validation_decision_stats"] = val_decision_stats_calibrated
    else:
        two_branch_info["validation_fusion_metrics"] = val_fused_metrics_fixed
        two_branch_info["validation_decision_stats"] = val_decision_stats_fixed

    threshold_tuning_info: Dict[str, Any] = {
        "mode": "two_branch_binary_cnn_with_ae_booster",
        "selection": {
            "active_policy": str(binary_booster_active_policy),
            "ae_attack_veto_q": float(binary_ae_booster_q_threshold),
            "cnn_binary_threshold": float(cls_threshold),
            "calibrator_enabled": bool(args.binary_booster_use_calibrator == 1),
            "calibrator_optimized": bool(binary_booster_calibrator_optimize),
            "calibrator_threshold": float(binary_booster_calibrator_threshold),
            "calibrator_max_normal_fpr": float(
                binary_booster_calibrator_info.get(
                    "max_normal_fpr_constraint",
                    args.binary_booster_calibrator_max_normal_fpr,
                )
            ),
            "calibrator_include_recon_stats": bool(
                binary_booster_calibrator_info.get(
                    "include_recon_feature_stats",
                    args.binary_booster_calibrator_include_recon_stats == 1,
                )
            ),
            "calibrator_temperature": float(binary_booster_calibrator_info.get("temperature", 1.0)),
        },
    }

    _, test_recon_feat, test_weighted_mse = reconstruction_error_features_and_weighted(
        model=autoencoder,
        X=X_test,
        feature_std=feature_std,
        device=device,
    )
    recon_errors_test = test_weighted_mse.astype(np.float32)
    q_scores_test = _percentile_scores_from_reference(normal_error_reference_sorted, recon_errors_test)
    test_recon_feat = np.concatenate([test_recon_feat, test_weighted_mse.reshape(-1, 1)], axis=1).astype(np.float32)
    if calibrator_residual_pca_test.shape[1] > 0:
        test_recon_feat = np.concatenate([test_recon_feat, calibrator_residual_pca_test], axis=1).astype(np.float32)
    prob_attack_test = predict_prob_attack(cnn, X_test_cnn_prepared, device=device)
    cnn_only_pred_test = (prob_attack_test >= float(cls_threshold)).astype(np.int64)
    cnn_only_eval_bundle = evaluate_binary(
        y_true=y_test,
        prob_attack=prob_attack_test,
        recon_errors_test=np.zeros_like(prob_attack_test),
        recon_threshold=0.0,
        unknown_attack_mask_test=labels.unknown_attack_mask_test,
        cls_threshold=float(cls_threshold),
        precomputed_pred=cnn_only_pred_test,
    )
    
    # Initialize test calibrated probability with raw value (may be overwritten by calibrator below)
    prob_attack_test_calibrated = prob_attack_test.astype(np.float32)

    y_pred_test_fixed, prob_attack_test_fixed, test_decision_stats_fixed = binary_cnn_with_ae_booster_predict(
        prob_attack=prob_attack_test,
        recon_errors=recon_errors_test,
        normal_error_reference_sorted=normal_error_reference_sorted,
        ae_attack_veto_q=float(binary_ae_booster_q_threshold),
        cls_threshold=float(cls_threshold),
    )
    eval_bundle_fixed = evaluate_binary(
        y_true=y_test,
        prob_attack=prob_attack_test_fixed,
        recon_errors_test=recon_errors_test,
        recon_threshold=recon_threshold,
        unknown_attack_mask_test=labels.unknown_attack_mask_test,
        cls_threshold=float(cls_threshold),
        precomputed_pred=y_pred_test_fixed,
    )

    two_branch_info["test_decision_stats_fixed_rule"] = test_decision_stats_fixed
    two_branch_info["test_evaluation_fixed_rule"] = evaluation_bundle_to_dict(eval_bundle_fixed)

    eval_bundle = eval_bundle_fixed
    if binary_booster_calibrator_model is not None:
        selected_calibrator_temperature = float(binary_booster_calibrator_info.get("temperature", 1.0))
        X_test_calibrator, q_scores_test_cal, entropy_test_cal = build_binary_booster_calibration_features(
            prob_attack=prob_attack_test,
            recon_errors=recon_errors_test,
            normal_error_reference_sorted=normal_error_reference_sorted,
            temperature=selected_calibrator_temperature,
        )
        if bool(binary_booster_calibrator_info.get("include_recon_feature_stats", False)):
            recon_feature_mean = np.asarray(binary_booster_calibrator_info.get("recon_feature_mean", []), dtype=np.float32)
            recon_feature_std = np.asarray(binary_booster_calibrator_info.get("recon_feature_std", []), dtype=np.float32)
            if (
                recon_feature_mean.ndim == 1
                and recon_feature_std.ndim == 1
                and recon_feature_mean.size == test_recon_feat.shape[1]
                and recon_feature_std.size == test_recon_feat.shape[1]
            ):
                recon_test_std = apply_feature_standardizer(
                    np.asarray(test_recon_feat, dtype=np.float32),
                    recon_feature_mean,
                    recon_feature_std,
                )
                X_test_calibrator = np.concatenate([X_test_calibrator, recon_test_std], axis=1).astype(np.float32)

        prob_attack_test_calibrated = predict_binary_booster_calibrator_prob(
            model=binary_booster_calibrator_model,
            X=X_test_calibrator,
            device=device,
        )
        y_pred_test_calibrated = (prob_attack_test_calibrated >= float(binary_booster_calibrator_threshold)).astype(np.int64)
        eval_bundle_calibrated = evaluate_binary(
            y_true=y_test,
            prob_attack=prob_attack_test_calibrated,
            recon_errors_test=recon_errors_test,
            recon_threshold=recon_threshold,
            unknown_attack_mask_test=labels.unknown_attack_mask_test,
            cls_threshold=float(binary_booster_calibrator_threshold),
            precomputed_pred=y_pred_test_calibrated,
        )
        two_branch_info["test_evaluation_calibrated"] = evaluation_bundle_to_dict(eval_bundle_calibrated)
        two_branch_info["test_decision_stats_calibrated"] = {
            "mean_prob_attack_feature": float(np.mean(X_test_calibrator[:, 0])),
            "mean_q_score": float(np.mean(q_scores_test_cal)),
            "mean_entropy": float(np.mean(entropy_test_cal)),
            "mean_calibrated_probability": float(np.mean(prob_attack_test_calibrated)),
            "selected_threshold": float(binary_booster_calibrator_threshold),
            "temperature": float(selected_calibrator_temperature),
            "pred_attack_rate": float(np.mean(y_pred_test_calibrated == 1)),
            "pred_normal_rate": float(np.mean(y_pred_test_calibrated == 0)),
        }

        if binary_booster_active_policy == "calibrated_mlp":
            eval_bundle = eval_bundle_calibrated
            two_branch_info["test_decision_stats"] = two_branch_info["test_decision_stats_calibrated"]
        else:
            two_branch_info["test_decision_stats"] = test_decision_stats_fixed
    else:
        two_branch_info["test_decision_stats"] = test_decision_stats_fixed

    print_eval(cnn_only_eval_bundle, title="Standalone CNN Results")
    print_eval(eval_bundle, title="Hybrid Pipeline Results")
    reference_comparison = build_reference_run_comparison(eval_bundle)
    print_reference_run_comparison(reference_comparison)

    final_prob_attack_test = (
        prob_attack_test_calibrated
        if binary_booster_active_policy == "calibrated_mlp" and binary_booster_calibrator_model is not None
        else prob_attack_test_fixed
    )
    final_pred_test = (
        y_pred_test_calibrated
        if binary_booster_active_policy == "calibrated_mlp" and y_pred_test_calibrated is not None
        else y_pred_test_fixed
    )
    unknown_attack_recall_by_type: Dict[str, float] = {}
    for attack_type in labels.unknown_attack_types:
        attack_mask = test_raw_labels == attack_type
        if np.any(attack_mask):
            unknown_attack_recall_by_type[attack_type] = float(np.mean(final_pred_test[attack_mask] == 1))
        else:
            unknown_attack_recall_by_type[attack_type] = 0.0

    print("\n=== Leakage Audit ===")
    split_source = "merged (stratified from KDDTrain+ + KDDTest+)" if bool(args.merge_official_splits == 1) else "KDDTrain+ / KDDTest+"
    print(f"Train/Test split source: {split_source}")
    print("Preprocessing fit scope: train only")
    print("Two-branch mode: enabled")
    print("Autoencoder fit scope: normal-only train split samples")
    print("SMOTE fit_resample scope: classifier branch train split features only")
    print("CNN fit scope: SMOTE-augmented classifier branch train split features (plus val monitoring from train)")
    print("Test used only for final transform/inference and metrics")

    fusion_info = {
        "enabled": bool(enable_residual_vector_fusion),
        "recon_feature_dim": int(residual_vector_dim),
        "recon_feature_weight": float(args.residual_vector_weight),
        "classifier_input_dim": int(X_train_cnn.shape[1]),
        "recon_feature_names": ["abs_residual_vector"] if enable_residual_vector_fusion else [],
        "standardization_scope": "train_split_only_classifier_branch",
        "residual_vector_fusion_enabled": bool(enable_residual_vector_fusion),
        "residual_vector_dim": int(residual_vector_dim),
        "residual_vector_weight": float(args.residual_vector_weight),
    }

    thresholds_payload: Dict[str, Any] = {
        "reconstruction_threshold": float(recon_threshold),
        "classifier_threshold": float(cls_threshold),
        "cnn_num_classes": int(cnn_num_classes),
        "enable_ae_latent_fusion": bool(enable_ae_latent_fusion),
        "two_branch_fusion_mode": "binary_ae_booster",
        "confidence_policy": "binary_cnn_with_ae_booster",
        "ae_attack_veto_q": float(binary_ae_booster_q_threshold),
        "binary_booster_active_policy": str(binary_booster_active_policy),
        "binary_booster_calibrator_enabled": bool(binary_booster_calibrator_model is not None),
        "binary_booster_calibrator_threshold": float(binary_booster_calibrator_threshold),
        "binary_booster_calibrator_optimized": bool(binary_booster_calibrator_optimize),
        "binary_booster_calibrator_max_normal_fpr": float(
            binary_booster_calibrator_info.get(
                "max_normal_fpr_constraint",
                args.binary_booster_calibrator_max_normal_fpr,
            )
        ),
        "binary_booster_calibrator_include_recon_stats": bool(
            binary_booster_calibrator_info.get(
                "include_recon_feature_stats",
                args.binary_booster_calibrator_include_recon_stats == 1,
            )
        ),
        "binary_booster_calibrator_temperature": float(binary_booster_calibrator_info.get("temperature", 1.0)),
        "reconstruction_error_metric": "weighted_mse",
        "feature_std_summary": {
            "min": float(np.min(feature_std)),
            "max": float(np.max(feature_std)),
            "mean": float(np.mean(feature_std)),
        },
    }

    return {
        "cnn_training_history": cnn_history,
        "cnn_only_evaluation": evaluation_bundle_to_dict(cnn_only_eval_bundle),
        "cnn_only_prob_test": prob_attack_test.astype(np.float32),
        "prob_attack_test": final_prob_attack_test.astype(np.float32),
        "recon_errors_test": recon_errors_test.astype(np.float32),
        "q_scores_test": q_scores_test.astype(np.float32),
        "unknown_attack_recall_by_type": unknown_attack_recall_by_type,
        "y_test": y_test,
        "binary_labels": {
            "y_val": y_val,
            "y_test": y_test,
            "unknown_attack_mask_test": labels.unknown_attack_mask_test,
        },
        "calibrated_probas": {
            "prob_val_calibrated": prob_attack_val_calibrated.astype(np.float32),
            "prob_test_calibrated": prob_attack_test_calibrated.astype(np.float32),
        },
        "dataset_summary": {
            "merge_official_splits": bool(args.merge_official_splits == 1),
            "merge_test_ratio": float(args.merge_test_ratio),
            "train_rows": int(dist["train_samples"]),
            "test_rows": int(dist["test_samples"]),
            "unknown_attack_type_count": int(dist["unknown_attack_type_count"]),
            "unknown_attack_record_count": int(dist["unknown_attack_record_count"]),
            "unknown_attack_types": labels.unknown_attack_types,
        },
        "smote_stats": smote_stats,
        "feature_fusion": fusion_info,
        "model_config": {
            "cnn_variant": effective_cnn_variant,
            "cnn_base_channels": int(args.cnn_base_channels),
            "cnn_stem_dim": int(args.cnn_stem_dim),
            "cnn_num_classes": int(cnn_num_classes),
            "enable_ae_latent_fusion": bool(enable_ae_latent_fusion),
            "enable_residual_vector_fusion": bool(enable_residual_vector_fusion),
            "residual_vector_weight": float(args.residual_vector_weight),
            "two_branch_system": bool(two_branch_enabled),
            "use_binary_cnn_with_ae_booster": bool(use_binary_cnn_with_ae_booster),
            "binary_booster_active_policy": str(binary_booster_active_policy),
            "binary_booster_use_calibrator": bool(args.binary_booster_use_calibrator == 1),
            "binary_booster_decision_policy": str(args.binary_booster_decision_policy),
            "binary_booster_calibrator_max_normal_fpr": float(args.binary_booster_calibrator_max_normal_fpr),
            "binary_booster_calibrator_threshold_steps": int(args.binary_booster_calibrator_threshold_steps),
            "binary_booster_calibrator_c": float(args.binary_booster_calibrator_c),
            "binary_booster_calibrator_class_weight_balanced": bool(
                args.binary_booster_calibrator_class_weight_balanced == 1
            ),
            "binary_booster_calibrator_include_recon_stats": bool(
                args.binary_booster_calibrator_include_recon_stats == 1
            ),
            "binary_booster_calibrator_optimize": bool(args.binary_booster_calibrator_optimize == 1),
            "binary_booster_calibrator_max_normal_fpr_grid": str(args.binary_booster_calibrator_max_normal_fpr_grid),
            "binary_booster_calibrator_include_recon_stats_grid": str(
                args.binary_booster_calibrator_include_recon_stats_grid
            ),
            "binary_booster_calibrator_temperature_grid": str(args.binary_booster_calibrator_temperature_grid),
            "binary_booster_calibrator_opt_objective": str(args.binary_booster_calibrator_opt_objective),
            "binary_booster_calibrator_opt_target_normal_fpr": float(
                args.binary_booster_calibrator_opt_target_normal_fpr
            ),
            "binary_booster_calibrator_opt_target_fpr_weight": float(
                args.binary_booster_calibrator_opt_target_fpr_weight
            ),
            "binary_booster_calibrator_hidden_dim": int(args.binary_booster_calibrator_hidden_dim),
            "binary_booster_calibrator_dropout": float(args.binary_booster_calibrator_dropout),
            "binary_booster_calibrator_epochs": int(args.binary_booster_calibrator_epochs),
            "binary_booster_calibrator_lr": float(args.binary_booster_calibrator_lr),
            "binary_booster_calibrator_patience": int(args.binary_booster_calibrator_patience),
            "binary_booster_calibrator_batch_size": int(args.binary_booster_calibrator_batch_size),
            "feature_std_summary": {
                "min": float(np.min(feature_std)),
                "max": float(np.max(feature_std)),
                "mean": float(np.mean(feature_std)),
            },
        },
        "two_branch": two_branch_info,
        "thresholds": thresholds_payload,
        "threshold_tuning": threshold_tuning_info,
        "evaluation": evaluation_bundle_to_dict(eval_bundle),
        "comparison_to_reference_runs": reference_comparison,
    }

## Plotting Functions

Automatic evaluation plots for training curves, confusion matrix, ROC/PR curves, and distribution diagnostics.

In [16]:
def generate_evaluation_plots(run_results: Dict[str, Any], run_dir: Path | str) -> None:
    output_dir = Path(run_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    def _save_figure(fig: plt.Figure, filename: str) -> None:
        fig.savefig(output_dir / filename, dpi=450, bbox_inches="tight")
        plt.close(fig)

    history = run_results.get("cnn_training_history", {})
    train_loss = list(history.get("train_loss", []))
    val_loss = list(history.get("val_loss", []))
    val_f1 = list(history.get("val_f1", []))
    train_acc = list(history.get("train_acc", []))
    val_acc = list(history.get("val_acc", []))
    best_epoch = int(history.get("best_epoch", 0))

    if train_loss and val_loss and train_acc and val_acc:
        epochs = np.arange(1, len(train_loss) + 1)
        fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

        axes[0].plot(epochs, train_loss, label="Train Loss", marker="o", markersize=3, linewidth=2)
        axes[0].plot(epochs, val_loss, label="Val Loss", marker="o", markersize=3, linewidth=2)
        if best_epoch > 0:
            axes[0].axvline(best_epoch, color="tab:red", linestyle="--", linewidth=1)
        axes[0].set_ylabel("Loss")
        axes[0].set_title("CNN Loss Curves")
        loss_vals = np.asarray(train_loss + val_loss, dtype=np.float32)
        if loss_vals.size > 0:
            loss_min = float(np.min(loss_vals))
            loss_max = float(np.max(loss_vals))
            pad = max(1e-6, 0.08 * (loss_max - loss_min))
            axes[0].set_ylim(loss_min - pad, loss_max + pad)
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].plot(epochs, train_acc, label="Train Accuracy", marker="o", markersize=3, linewidth=2)
        axes[1].plot(epochs, val_acc, label="Val Accuracy", marker="o", markersize=3, linewidth=2)
        if best_epoch > 0:
            axes[1].axvline(best_epoch, color="tab:red", linestyle="--", linewidth=1, label="Best Epoch")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        acc_vals = np.asarray(train_acc + val_acc, dtype=np.float32)
        if acc_vals.size > 0:
            acc_min = float(np.min(acc_vals))
            acc_max = float(np.max(acc_vals))
            pad = max(1e-4, 0.08 * (acc_max - acc_min))
            axes[1].set_ylim(max(0.0, acc_min - pad), min(1.0, acc_max + pad))
        axes[1].set_title("CNN Accuracy Curves")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        _save_figure(fig, "01_cnn_training.png")

    if val_f1:
        epochs = np.arange(1, len(val_f1) + 1)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(epochs, val_f1, label="Val Macro-F1", marker="o", markersize=3, linewidth=2, color="tab:green")
        if best_epoch > 0:
            ax.axvline(best_epoch, color="tab:red", linestyle="--", linewidth=1, label="Best Epoch")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Macro-F1")
        ax.set_title("CNN Validation Macro-F1")
        f1_vals = np.asarray(val_f1, dtype=np.float32)
        if f1_vals.size > 0:
            f1_min = float(np.min(f1_vals))
            f1_max = float(np.max(f1_vals))
            pad = max(1e-4, 0.08 * (f1_max - f1_min))
            ax.set_ylim(max(0.0, f1_min - pad), min(1.0, f1_max + pad))
        ax.grid(True, alpha=0.3)
        ax.legend()
        _save_figure(fig, "01b_cnn_val_f1.png")

    eval_dict = run_results.get("evaluation", {})
    confusion = eval_dict.get("confusion_matrix")
    if confusion is not None:
        cm = np.asarray(confusion, dtype=np.int64)
        if cm.shape == (2, 2):
            total = float(np.sum(cm))
            percent = (cm / total) * 100.0 if total > 0.0 else np.zeros_like(cm, dtype=np.float32)
            annot = np.empty_like(cm, dtype=object)
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    annot[i, j] = f"{cm[i, j]}\n{percent[i, j]:.1f}%"

            fig, ax = plt.subplots(figsize=(6.5, 5.5))
            sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=True, square=True, ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("Actual")
            ax.set_xticklabels(["Normal", "Attack"])
            ax.set_yticklabels(["Normal", "Attack"], rotation=0)
            ax.set_title("Confusion Matrix")
            _save_figure(fig, "02_confusion_matrix.png")

    y_test = run_results.get("y_test")
    prob_attack = run_results.get("prob_attack_test")
    cnn_only_prob = run_results.get("cnn_only_prob_test")
    if y_test is not None and prob_attack is not None:
        y_true = np.asarray(y_test, dtype=np.int64)
        y_score = np.asarray(prob_attack, dtype=np.float32)
        y_score_cnn = None
        if cnn_only_prob is not None:
            y_score_cnn = np.asarray(cnn_only_prob, dtype=np.float32)

        if y_true.size == y_score.size and len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = roc_auc_score(y_true, y_score)
            precision, recall, _ = precision_recall_curve(y_true, y_score)
            pr_auc = average_precision_score(y_true, y_score)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            axes[0].plot(fpr, tpr, label=f"Hybrid AUC={roc_auc:.4f}")
            if y_score_cnn is not None and y_score_cnn.size == y_true.size:
                fpr_cnn, tpr_cnn, _ = roc_curve(y_true, y_score_cnn)
                roc_auc_cnn = roc_auc_score(y_true, y_score_cnn)
                axes[0].plot(fpr_cnn, tpr_cnn, label=f"CNN AUC={roc_auc_cnn:.4f}")
            axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
            axes[0].set_xlabel("False Positive Rate")
            axes[0].set_ylabel("True Positive Rate")
            axes[0].set_title("ROC Curve")
            axes[0].legend()

            axes[1].plot(recall, precision, label=f"Hybrid AP={pr_auc:.4f}", color="tab:orange")
            if y_score_cnn is not None and y_score_cnn.size == y_true.size:
                precision_cnn, recall_cnn, _ = precision_recall_curve(y_true, y_score_cnn)
                pr_auc_cnn = average_precision_score(y_true, y_score_cnn)
                axes[1].plot(recall_cnn, precision_cnn, label=f"CNN AP={pr_auc_cnn:.4f}")
            axes[1].set_xlabel("Recall")
            axes[1].set_ylabel("Precision")
            axes[1].set_title("Precision-Recall Curve")
            axes[1].legend()
            _save_figure(fig, "03_roc_pr_curve.png")

    if eval_dict:
        accuracy_vals = [
            float(eval_dict.get("accuracy", 0.0)),
            float(eval_dict.get("known_subset_accuracy", 0.0)),
            float(eval_dict.get("unknown_subset_accuracy", 0.0)),
        ]
        labels = ["Overall", "Known", "Unknown"]
        x = np.arange(len(labels))

        fig, ax = plt.subplots(figsize=(7.5, 5))
        ax.bar(x, accuracy_vals, color=["tab:blue", "tab:green", "tab:red"], alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0.0, 1.0)
        ax.set_title("Known vs. Unknown Accuracy Breakdown")
        _save_figure(fig, "04_accuracy_breakdown.png")

    cnn_only_eval = run_results.get("cnn_only_evaluation", {})
    if eval_dict and cnn_only_eval:
        metric_labels = ["Accuracy", "Macro F1", "Attack Recall", "Unknown Recall", "Normal FPR"]
        cnn_values = [
            float(cnn_only_eval.get("accuracy", 0.0)),
            float(cnn_only_eval.get("macro_f1", 0.0)),
            float(cnn_only_eval.get("attack_recall", 0.0)),
            float(cnn_only_eval.get("unknown_attack_recall", 0.0)),
            float(cnn_only_eval.get("normal_false_positive_rate", 0.0)),
        ]
        hybrid_values = [
            float(eval_dict.get("accuracy", 0.0)),
            float(eval_dict.get("macro_f1", 0.0)),
            float(eval_dict.get("attack_recall", 0.0)),
            float(eval_dict.get("unknown_attack_recall", 0.0)),
            float(eval_dict.get("normal_false_positive_rate", 0.0)),
        ]
        x = np.arange(len(metric_labels))
        width = 0.38

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x - width / 2.0, cnn_values, width, label="CNN Only", color="tab:blue", alpha=0.85)
        ax.bar(x + width / 2.0, hybrid_values, width, label="Hybrid", color="tab:green", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels)
        ax.set_ylabel("Score")
        ax.set_ylim(0.0, 1.0)
        ax.set_title("CNN vs. Hybrid Performance")
        ax.legend()
        _save_figure(fig, "09_cnn_vs_hybrid_metrics.png")

    recon_errors = run_results.get("recon_errors_test")
    q_scores = run_results.get("q_scores_test")
    unknown_mask = run_results.get("binary_labels", {}).get("unknown_attack_mask_test")
    if recon_errors is not None and y_test is not None and unknown_mask is not None:
        recon = np.asarray(recon_errors, dtype=np.float32)
        y_true = np.asarray(y_test, dtype=np.int64)
        unknown_mask_arr = np.asarray(unknown_mask, dtype=bool)
        if recon.size == y_true.size == unknown_mask_arr.size:
            normal_mask = y_true == 0
            attack_mask = y_true == 1
            known_attack_mask = attack_mask & (~unknown_mask_arr)
            unknown_attack_mask = attack_mask & unknown_mask_arr

            fig, ax = plt.subplots(figsize=(9, 5))
            sns.histplot(
                recon[normal_mask],
                bins=50,
                stat="density",
                color="tab:blue",
                alpha=0.5,
                label="Normal",
                ax=ax,
            )
            sns.histplot(
                recon[known_attack_mask],
                bins=50,
                stat="density",
                color="tab:orange",
                alpha=0.5,
                label="Known Attack",
                ax=ax,
            )
            sns.histplot(
                recon[unknown_attack_mask],
                bins=50,
                stat="density",
                color="tab:green",
                alpha=0.5,
                label="Unknown Attack",
                ax=ax,
            )
            ax.set_xlabel("AE Reconstruction Error (weighted MSE)")
            ax.set_ylabel("Density")
            ax.set_title("Reconstruction Error Distribution")
            ax.legend()
            _save_figure(fig, "05_recon_error_distribution.png")

    if q_scores is not None and y_test is not None and unknown_mask is not None:
        q_vals = np.asarray(q_scores, dtype=np.float32)
        y_true = np.asarray(y_test, dtype=np.int64)
        unknown_mask_arr = np.asarray(unknown_mask, dtype=bool)
        if q_vals.size == y_true.size == unknown_mask_arr.size:
            normal_mask = y_true == 0
            attack_mask = y_true == 1
            known_attack_mask = attack_mask & (~unknown_mask_arr)
            unknown_attack_mask = attack_mask & unknown_mask_arr

            fig, ax = plt.subplots(figsize=(9, 5))
            sns.histplot(
                q_vals[normal_mask],
                bins=50,
                stat="density",
                color="tab:blue",
                alpha=0.5,
                label="Normal",
                ax=ax,
            )
            sns.histplot(
                q_vals[known_attack_mask],
                bins=50,
                stat="density",
                color="tab:orange",
                alpha=0.5,
                label="Known Attack",
                ax=ax,
            )
            sns.histplot(
                q_vals[unknown_attack_mask],
                bins=50,
                stat="density",
                color="tab:green",
                alpha=0.5,
                label="Unknown Attack",
                ax=ax,
            )
            ax.set_xlabel("AE Q-Score (percentile vs. normal validation)")
            ax.set_ylabel("Density")
            ax.set_title("Q-Score Distribution")
            ax.legend()
            _save_figure(fig, "06_q_score_distribution.png")

    recall_by_type = run_results.get("unknown_attack_recall_by_type", {})
    if recall_by_type:
        names = list(recall_by_type.keys())
        values = np.asarray([recall_by_type[name] for name in names], dtype=np.float32)
        order = np.argsort(values)
        names_sorted = [names[i] for i in order]
        values_sorted = values[order]

        fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(names_sorted))))
        ax.barh(names_sorted, values_sorted, color="tab:purple", alpha=0.85)
        ax.set_xlabel("Recall")
        ax.set_xlim(0.0, 1.0)
        ax.set_title("Per-Unknown-Attack Recall (low to high)")
        _save_figure(fig, "07_unknown_attack_recall.png")

    calibrator_info = (
        run_results.get("two_branch", {})
        .get("binary_booster_calibrator", {})
        .get("optimization", {})
    )
    candidates = calibrator_info.get("candidates", [])
    if candidates:
        acc = np.array([c.get("accuracy", 0.0) for c in candidates], dtype=np.float32)
        attack_recall = np.array([c.get("attack_recall", 0.0) for c in candidates], dtype=np.float32)
        fpr = np.array([c.get("normal_fpr", 0.0) for c in candidates], dtype=np.float32)
        bins = np.array([0.0, 0.08, 0.12, 0.16, 0.2, 1.0], dtype=np.float32)
        bin_ids = np.digitize(fpr, bins, right=False)

        fig, ax = plt.subplots(figsize=(7.5, 6))
        scatter = ax.scatter(acc, attack_recall, c=bin_ids, cmap="viridis", alpha=0.85)
        ax.set_xlabel("Accuracy")
        ax.set_ylabel("Attack Recall")
        ax.set_title("Calibrator Threshold Sweep")
        cbar = fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("FPR Bin")
        _save_figure(fig, "08_calibrator_sweep.png")

    plt.close("all")

## Ensemble Setup (Optional)

Optional ensemble averaging over multiple seeds, with aggregated validation threshold selection and test evaluation.

In [17]:
def _execute_with_args(args: argparse.Namespace) -> Dict[str, Any]:
    args.dataset_dir = str(resolve_dataset_dir(args.dataset_dir))
    args.output_dir = str(resolve_output_dir(args.output_dir))
    
    # Check if ensemble mode is enabled
    ensemble_seeds_str = str(getattr(args, 'ensemble_seeds', '')).strip()
    if ensemble_seeds_str:
        # Parse ensemble seeds
        try:
            seed_list = [int(s.strip()) for s in ensemble_seeds_str.split(",") if s.strip()]
        except ValueError:
            raise ValueError(f"Invalid --ensemble-seeds value: {ensemble_seeds_str}. Expected comma-separated integers.")
        
        if len(seed_list) > 1:
            print(f"\n[Ensemble] Running ensemble with {len(seed_list)} seeds: {seed_list}")
            all_prob_val = []
            all_prob_test = []
            run_results_last = None
            
            for i, seed in enumerate(seed_list):
                print(f"\n[Ensemble] Seed {i+1}/{len(seed_list)}: {seed}")
                args.seed = seed
                artifacts = create_run_artifacts(Path(args.output_dir))
                save_json(artifacts.config_path, vars(args))
                
                with artifacts.log_path.open("w", encoding="utf-8") as log_handle:
                    tee = TeeStream(sys.stdout, log_handle)
                    with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
                        try:
                            run_results = run_pipeline(args)
                            run_results["status"] = "completed"
                        except KeyboardInterrupt:
                            print("\n[Ensemble] Interrupted by user (KeyboardInterrupt).")
                            raise

                        try:
                            generate_evaluation_plots(run_results, artifacts.run_dir)
                        except Exception as exc:
                            print(f"[Plots] Failed to generate evaluation plots: {exc}")
                        
                        run_results["run_artifacts"] = {
                            "run_id": artifacts.run_id,
                            "run_dir": artifacts.run_dir,
                            "log_path": artifacts.log_path,
                            "results_path": artifacts.results_path,
                            "config_path": artifacts.config_path,
                        }
                        save_json(artifacts.results_path, run_results)
                
                # Collect calibrated probabilities
                calibrated = run_results["calibrated_probas"]
                all_prob_val.append(np.asarray(calibrated["prob_val_calibrated"], dtype=np.float32))
                all_prob_test.append(np.asarray(calibrated["prob_test_calibrated"], dtype=np.float32))
                run_results_last = run_results
            
            # Average probabilities
            avg_prob_val = np.mean(all_prob_val, axis=0).astype(np.float32)
            avg_prob_test = np.mean(all_prob_test, axis=0).astype(np.float32)
            
            # Extract labels from last run
            y_val = run_results_last["binary_labels"]["y_val"]
            y_test = run_results_last["binary_labels"]["y_test"]
            unknown_mask = run_results_last["binary_labels"]["unknown_attack_mask_test"]
            
            print("\n[Ensemble] Computing ensemble evaluation...")
            
            # Select threshold from averaged validation probabilities
            threshold_info = select_constrained_binary_threshold(
                y_true=y_val,
                prob_attack=avg_prob_val,
                max_normal_fpr=0.12,
                steps=401,
            )
            best_threshold = float(threshold_info["threshold"])
            
            print(f"[Ensemble] Selected threshold: {best_threshold:.4f} (max_normal_fpr=0.12)")
            print(f"[Ensemble] Feasible candidates: {int(threshold_info['feasible_candidate_count'])}")
            
            # Evaluate ensemble on test
            y_pred_ensemble = (avg_prob_test >= best_threshold).astype(np.int64)
            eval_ensemble = evaluate_binary(
                y_true=y_test,
                prob_attack=avg_prob_test,
                recon_errors_test=np.zeros_like(avg_prob_test),  # Not used when precomputed_pred provided
                recon_threshold=0.0,  # Not used
                unknown_attack_mask_test=unknown_mask,
                cls_threshold=best_threshold,
                precomputed_pred=y_pred_ensemble,
            )
            
            print("\n=== Ensemble Averaged Test Metrics ===")
            print_eval(eval_ensemble)
            ensemble_comparison = build_reference_run_comparison(eval_ensemble)
            print_reference_run_comparison(ensemble_comparison)
            
            # Prepare ensemble result
            ensemble_result = {
                "status": "completed",
                "ensemble_mode": True,
                "ensemble_seeds": seed_list,
                "ensemble_seed_count": len(seed_list),
                "average_calibrated_probabilities": {
                    "prob_val": avg_prob_val.astype(float).tolist() if avg_prob_val.size < 100000 else "too_large",
                    "prob_test": avg_prob_test.astype(float).tolist() if avg_prob_test.size < 100000 else "too_large",
                },
                "ensemble_threshold": float(best_threshold),
                "ensemble_threshold_info": threshold_info,
                "ensemble_evaluation": evaluation_bundle_to_dict(eval_ensemble),
                "ensemble_comparison_to_reference": ensemble_comparison,
                "individual_runs": [run_results_last],  # Could store all runs, but that's expensive
            }
            ensemble_artifacts = create_run_artifacts(Path(args.output_dir))
            ensemble_plot_payload = {
                "evaluation": evaluation_bundle_to_dict(eval_ensemble),
                "y_test": y_test,
                "prob_attack_test": avg_prob_test,
                "binary_labels": {
                    "unknown_attack_mask_test": unknown_mask,
                },
            }
            try:
                generate_evaluation_plots(ensemble_plot_payload, ensemble_artifacts.run_dir)
            except Exception as exc:
                print(f"[Plots] Failed to generate ensemble plots: {exc}")

            ensemble_result["run_artifacts"] = {
                "run_id": ensemble_artifacts.run_id,
                "run_dir": ensemble_artifacts.run_dir,
                "log_path": ensemble_artifacts.log_path,
                "results_path": ensemble_artifacts.results_path,
                "config_path": ensemble_artifacts.config_path,
            }
            save_json(ensemble_artifacts.results_path, ensemble_result)
            return ensemble_result
    
    # Single run (non-ensemble) path
    artifacts = create_run_artifacts(Path(args.output_dir))
    save_json(artifacts.config_path, vars(args))

    with artifacts.log_path.open("w", encoding="utf-8") as log_handle:
        tee = TeeStream(sys.stdout, log_handle)
        with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
            print(f"[Run] Artifact directory: {artifacts.run_dir}")
            print(f"[Run] Config file: {artifacts.config_path}")

            try:
                run_results = run_pipeline(args)
                run_results["status"] = "completed"
            except KeyboardInterrupt:
                print("\n[Run] Interrupted by user (KeyboardInterrupt).")
                run_results = {
                    "status": "interrupted",
                    "message": "Run interrupted before completion.",
                }

            if run_results.get("status") == "completed":
                try:
                    generate_evaluation_plots(run_results, artifacts.run_dir)
                except Exception as exc:
                    print(f"[Plots] Failed to generate evaluation plots: {exc}")

            run_results["run_artifacts"] = {
                "run_id": artifacts.run_id,
                "run_dir": artifacts.run_dir,
                "log_path": artifacts.log_path,
                "results_path": artifacts.results_path,
                "config_path": artifacts.config_path,
            }
            save_json(artifacts.results_path, run_results)
            print(f"[Run] Test results saved to: {artifacts.results_path}")
            print(f"[Run] Training log saved to: {artifacts.log_path}")

    print(f"Saved training log: {artifacts.log_path}")
    print(f"Saved test results: {artifacts.results_path}")
    return run_results

## Notebook Execution Entry Point

Notebook-friendly entry points plus the CLI `main` and `__main__` guard. The example call is commented out for safety.

In [18]:
def run_from_notebook(**overrides: Any) -> Dict[str, Any]:
    """
    Notebook-friendly runner that avoids argparse issues in ipykernel.

    Example:
        run_from_notebook(
            dataset_dir="/kaggle/input/datasets/hassan06/nslkdd",
            output_dir="/kaggle/working/training_runs",
            seed=42,
            two_branch_system=1,
            use_binary_cnn_with_ae_booster=1,
            binary_booster_use_calibrator=1,
            binary_booster_calibrator_optimize=1,
            binary_booster_decision_policy="calibrated",
            binary_booster_calibrator_max_normal_fpr=0.12,
            binary_booster_calibrator_max_normal_fpr_grid="0.10,0.12,0.14,0.16",
            binary_booster_calibrator_threshold_steps=401,
            binary_booster_calibrator_c=1.0,
            binary_booster_calibrator_class_weight_balanced=1,
            binary_booster_calibrator_include_recon_stats=1,
            binary_booster_calibrator_include_recon_stats_grid="1",
            binary_booster_calibrator_temperature_grid="1.0",
            binary_booster_calibrator_opt_objective="recall_balanced",
            binary_booster_calibrator_opt_target_normal_fpr=0.12,
            binary_booster_calibrator_opt_target_fpr_weight=0.5,
            enable_ae_latent_fusion=1,
            enable_residual_vector_fusion=1,
            residual_vector_weight=1.0,
            cnn_num_classes=2,
            cnn_variant="multiscale_stem",
            latent_dim=64,
            ae_epochs=30,
            ae_patience=30,
            cnn_epochs=30,
            cnn_patience=30,
            binary_booster_calibrator_hidden_dim=16,
            binary_booster_calibrator_dropout=0.3,
            binary_booster_calibrator_epochs=80,
            binary_booster_calibrator_lr=5e-4,
            binary_booster_calibrator_patience=10,
            binary_booster_calibrator_batch_size=1024,
        )
    """
    parser = build_arg_parser()
    args = parser.parse_args([])

    for key, value in overrides.items():
        attr = str(key).replace("-", "_")
        if not hasattr(args, attr):
            valid = ", ".join(sorted(vars(args).keys()))
            raise ValueError(f"Unknown notebook override: {key}. Valid args: {valid}")
        setattr(args, attr, value)

    return _execute_with_args(args)


def main(argv: List[str] | None = None) -> Dict[str, Any]:
    parser = build_arg_parser()

    if argv is not None:
        args = parser.parse_args(argv)
    elif "ipykernel" in sys.modules:
        # In notebooks, kernel injects extra args (e.g., -f <kernel.json>).
        args, unknown = parser.parse_known_args()
        if unknown:
            print(f"[Args] Ignoring notebook/kernel args: {' '.join(unknown)}")
    else:
        args = parser.parse_args()

    return _execute_with_args(args)


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        print("[Notebook] Definitions loaded. Use run_from_notebook(...) to start training.")
    else:
        main()

# Example:
# run_from_notebook(dataset_dir="/path/to/NSL-KDD", output_dir="training_runs", seed=42)

[Notebook] Definitions loaded. Use run_from_notebook(...) to start training.


## Merged-split

with 80/20 train test split

In [19]:
results = run_from_notebook(
    dataset_dir="/kaggle/input/datasets/hassan06/nslkdd",
    output_dir="/kaggle/working/training_runs",
    seed=42,

    merge_official_splits=1,
    merge_test_ratio=0.2,
    
    #val_seed=42,          # FIXED for all ensemble seeds
    #ensemble_seeds="42,123,456",


    two_branch_system=1,
    use_binary_cnn_with_ae_booster=1,
    binary_booster_use_calibrator=1,
    binary_booster_decision_policy="calibrated",

    enable_ae_latent_fusion=1,
    enable_residual_vector_fusion=1,
    residual_vector_weight=2.5,
    cnn_num_classes=2,
    cnn_variant="multiscale_stem",
    latent_dim=64,

    ae_epochs=40,
    ae_patience=10,
    cnn_epochs=40,                # can increase to 40 later
    cnn_patience=10,
    cnn_lr=5e-4,                  # NEW: slightly reduced

    binary_booster_calibrator_optimize=1,
    binary_booster_calibrator_opt_objective="accuracy",    # CHANGED from recall_balanced
    binary_booster_calibrator_opt_target_normal_fpr=0.10,   # keep tight FPR
    binary_booster_calibrator_opt_target_fpr_weight=0.5,
    binary_booster_calibrator_max_normal_fpr_grid="0.10,0.12,0.14,0.16",
    binary_booster_calibrator_include_recon_stats_grid="1",
    binary_booster_calibrator_temperature_grid="1.0",
    binary_booster_calibrator_threshold_steps=401,
    binary_booster_calibrator_c=1.0,
    binary_booster_calibrator_class_weight_balanced=1,

    binary_booster_calibrator_hidden_dim=32,   # NEW: more capacity
    binary_booster_calibrator_dropout=0.3,
    binary_booster_calibrator_epochs=80,
    binary_booster_calibrator_lr=5e-4,
    binary_booster_calibrator_patience=10,
    binary_booster_calibrator_batch_size=1024,
)

[Run] Artifact directory: /kaggle/working/training_runs/run_20260504_021606_970926
[Run] Config file: /kaggle/working/training_runs/run_20260504_021606_970926/run_config.json
[Data] Dataset directory: /kaggle/input/datasets/hassan06/nslkdd
[Data] Using merged-official-splits mode with test ratio 0.2
=== Split and Unknown Attack Summary ===
Train rows: 118813
Test rows:  29704
Unknown attack types in test: 0
Unknown attack records in test: 0

Using device: cuda
[Mode] AE latent fusion enabled: concatenating AE latent vectors to CNN inputs.
[Mode] Residual vector fusion enabled: concatenating standardized |X - AE(X)| vectors to CNN input (weight=2.500).
[Mode] Binary CNN + AE booster enabled: CNN is binary (normal/attack) and AE forces attack when q >= 0.96.
[Mode] Calibrator-only optimization enabled: fpr_grid=[0.1, 0.12, 0.14, 0.16], recon_stats_grid=[1], temperature_grid=[1.0], objective=accuracy, target_fpr=0.100, target_fpr_weight=0.500.
[AE] Epoch 001 | train_mse=0.133408 | val_mse

## Strict split

Train on kddtrain+, Test on kddtest+

In [20]:
results = run_from_notebook(
    dataset_dir="/kaggle/input/datasets/hassan06/nslkdd",
    output_dir="/kaggle/working/training_runs",
    seed=42,

    merge_official_splits=0,

    two_branch_system=1,
    use_binary_cnn_with_ae_booster=1,
    binary_booster_use_calibrator=1,
    binary_booster_decision_policy="calibrated",

    enable_ae_latent_fusion=1,
    enable_residual_vector_fusion=1,
    residual_vector_weight=1.0,
    cnn_num_classes=2,
    cnn_variant="multiscale_stem",
    latent_dim=64,

    ae_epochs=30,
    ae_patience=30,
    cnn_epochs=30,
    cnn_patience=30,

    binary_booster_calibrator_optimize=1,
    binary_booster_calibrator_opt_objective="recall_balanced",   # Balances recall vs FPR
    binary_booster_calibrator_opt_target_normal_fpr=0.12,
    binary_booster_calibrator_opt_target_fpr_weight=0.5,
    binary_booster_calibrator_max_normal_fpr_grid="0.10,0.12,0.14,0.16",
    binary_booster_calibrator_include_recon_stats_grid="1",      # Always include enhanced stats
    binary_booster_calibrator_temperature_grid="1.0",            # Keep temp=1 for simplicity
    binary_booster_calibrator_threshold_steps=401,
    binary_booster_calibrator_c=1.0,
    binary_booster_calibrator_class_weight_balanced=1,

    # MLP hyperparameters (new)
    binary_booster_calibrator_hidden_dim=32,
    binary_booster_calibrator_dropout=0.3,
    binary_booster_calibrator_epochs=200,
    binary_booster_calibrator_lr=5e-4,
    binary_booster_calibrator_patience=200,
    binary_booster_calibrator_batch_size=1024,
)

[Run] Artifact directory: /kaggle/working/training_runs/run_20260504_022940_577094
[Run] Config file: /kaggle/working/training_runs/run_20260504_022940_577094/run_config.json
[Data] Dataset directory: /kaggle/input/datasets/hassan06/nslkdd
[Data] Using official KDDTrain+/KDDTest+ split
=== Split and Unknown Attack Summary ===
Train rows: 125973
Test rows:  22544
Unknown attack types in test: 17
Unknown attack records in test: 3750

Using device: cuda
[Mode] AE latent fusion enabled: concatenating AE latent vectors to CNN inputs.
[Mode] Residual vector fusion enabled: concatenating standardized |X - AE(X)| vectors to CNN input (weight=1.000).
[Mode] Binary CNN + AE booster enabled: CNN is binary (normal/attack) and AE forces attack when q >= 0.96.
[Mode] Calibrator-only optimization enabled: fpr_grid=[0.1, 0.12, 0.14, 0.16], recon_stats_grid=[1], temperature_grid=[1.0], objective=recall_balanced, target_fpr=0.120, target_fpr_weight=0.500.
[AE] Epoch 001 | train_mse=0.156132 | val_mse=0.

In [21]:
!zip -r /kaggle/working/myfolder.zip /kaggle/working/

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 80%)
  adding: kaggle/working/training_runs/ (stored 0%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/ (stored 0%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/run_config.json (deflated 66%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/train.log (deflated 73%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/01_cnn_training.png (deflated 14%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/09_cnn_vs_hybrid_metrics.png (deflated 26%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/04_accuracy_breakdown.png (deflated 26%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/05_recon_error_distribution.png (deflated 24%)
  adding: kaggle/working/training_runs/run_20260504_021606_970926/06_q